# 手順5：シフト構造の本実装（構造 B・A・C × H の扱い (1)(2)(4)）

CLAUDE.md §5・§7。手順5a で決まった構造（**本命 B**：横・縦を別々にシフト、**対案 A**：縦だけシフト、基準 **C**：構造なしの通常の GPR）を、手順3e で決めた **H の扱い (1)・(2)・(4) の組み合わせ**で、同じ形で学習・評価する。データは、手順3d の 2 つの版（**5data 版**：5data で学習して 3data で外挿を検証／**Alldata 版**：全条件で学習して交差検証）を `DATA_VERSIONS` で切り替える。

**モデルの表**（記号は「構造_扱い」。例：B_1、A_2、C_4。手順3b の B0・B2 とは別物）

u = ln γ̇（条件平均）＋ ln a(Tset; E_h)、w = ln Fo = −(ln γ̇ ＋ 3 ln H) ＋ 定数、p = ln P3_2（条件平均、Pa）。ln a(T; E) = (E/R)(1/T − 1/Tref)、Tref = 483.15 K（210 ℃）。f・m・c は GP。

| H の扱い | 構造 B・A の形（A は E_h = 0） | 構造 C の形（Tset を入力に並べる） |
|---|---|---|
| (1) H を残す（`"1"`） | ln η = ln a(E_v) ＋ f(u, w) | f(ln γ̇, ln H, Tset)　※手順3b・3d の R3 |
| (2) 分ける（`"2"`） | ln a(E_v) ＋ **m(u)** ＋ **c(w)** | m(ln γ̇, Tset) ＋ c(w) |
| (4) P3_2 を入れる・H なし（`"4"`） | ln a(E_v) ＋ f(u, p) | f(ln γ̇, Tset, p)　※手順3e の W2 |
| (1+4)（`"14"`） | ln a(E_v) ＋ f(u, w, p) | f(ln γ̇, ln H, Tset, p) |
| (2+4)（`"24"`） | ln a(E_v) ＋ m(u, p) ＋ c(w) | m(ln γ̇, Tset, p) ＋ c(w) |

**(2) の意味**：Cross-WLF のように「材料の部分 m（せん断速度と温度の関数）」と、「厚み・冷却の補正 c(Fo)」を足し算で分ける。足し算のカーネル（m の部分と c の部分に、それぞれ Matern ＋ 線形項）で表す。m と c は定数だけ入れ替えられるので、図では c の平均を 0 にそろえる。

**(4) の注意**：P3_2 は流れの結果で、成形条件から予測するときは**事前に分からない**。しかも**厚みでほぼ決まる量**（ln P3_2 ≈ −1.5 ln H）なので、H を P3_2 の形で入れただけ、かもしれない（手順3e）。そこで (4) を含むモデルは 2 通りで評価する。
- **実測の P3_2**：検証の条件の、実測の条件平均を使う（P3_2 が分かっている場合の上限。手順3e と同じ）。
- **予測した P3_2**：事前に分からないとして、ln P3_2 を (ln γ̇, ln H, Tset) から GPR で予測し（学習の条件の点で学習）、その予測分布から `P32_DRAWS` 回引いて ln η を予測し、平均と分散にまとめる。P3_2 を予測するのに H が要るので、「実測」との差が、「(4) の良さが、P3_2 が分かっているおかげか」を表す。

**推定**（これまでどおり scikit-learn）
1. **E はプロファイル尤度で決める**：条件平均の点で、E の格子の各点について GPR を学習し、対数周辺尤度（LML）が最大の E を採る。LML が最大から 1.92 以内の範囲を、おおよその 95% 区間（B は斜めの帯）とする。粗い格子で全体を見て、最良の点の近くを細かい格子で計算し直す。前の格子点の解を初期値にして、局所解でのがたつきを減らす。出力の自動標準化は使わない（E ごとに尺度が変わり、尤度が比べられなくなるため）。
2. **評価用のモデルは、ショットごとに学習し直す**：最良の E で、入力の γ̇ を条件平均にして学習する（成形条件から見たばらつきを出すため）。E は、学習データごとに、その学習データだけで推定する（CV の分割ごとにも推定する。検証の情報は使わない）。
3. **B の E_h と E_v は、別々には決まりにくい**（手順5a：差 E_v − E_h だけが決まる）。両方を推定し、斜めの帯と差を示す。設定 `B_EH_FIX` に数を入れると、E_h を固定して実行し直せる。

**MCMC は、結果を見てから別の手順で行う**（ユーザー決定、10-14）。

## 使い方（このノートを一人で実行して、結果を判断するために）

1. メニューの「すべて実行」で、上から順に実行する（約 150 分）。カーネルは、このプロジェクトの `.venv` を選ぶ。
2. 設定を変えたいときは、最初の「設定エリア」のセルだけを直して、最初から実行し直す（高解像度のグラフの設定は、下の 7）。
3. 各節の最後に、**判定**が出る（`>> 判定：OK` か `>> 判定：★要確認`）。OK は「想定どおり」、★ は「想定と違う、または結果を読むときに注意が要る点」。基準は、判定の文の中に書いてある。★ が出ても、エラーではない。説明書の「判定の読み方」で、★ の意味を確かめる。
4. 図と表は `results/step5/` に保存される。図の見方は、説明書 `results/学習結果説明_手順5.pdf` にある。
5. データを取り直したとき、問題のあるデータを入れ替えたときは、`問題データ一覧.md` の手順に従う。このノートは、全条件のファイルを読み、`02_機械学習データ/問題データ一覧.csv` の「除外」を除く。
6. 用語や図の見方で分からないところは、説明書の用語集で調べる。記号（Tff、Tset、R3、E_h など）の意味は、`results/学習結果説明_記号表.pdf` に一覧がある。
7. **高解像度のグラフ**（報告書・発表資料用。GPR6月ノートのセル9-2 と同じ見せ方）は、「高解像度のグラフ」の節のセルで作る。**このセルだけは、セルの先頭にも設定エリアがあり、そこだけを直して、このセルだけを実行し直せる**（学習はやり直さない。全体を一度実行した後で）。保存先は `results/step5/highres/`（Git には入れない）。

### 実行時間の目安と、短くする方法

全部（2 つの版、3 構造 × 5 通り）で約 2〜3 時間（プロファイル尤度が約 1 時間、ショットごとの学習が約 1〜1.5 時間）。短くするには、設定エリアで次を減らす。
- `DATA_VERSIONS = ["5data"]`：約 20 分。
- `SPLITS`：Alldata 版の `"CV"`（10 分割）を外す。
- `H_VARIANTS`（例：`["1", "2"]`）、`STRUCTS`（例：`["B", "C"]`）。

「すべて実行」でなく、版ごとに分かれたセル（プロファイル・学習）を、順に実行してもよい。1 セルが長いので、メニューの「すべて実行」では時間の上限（ノートの設定）に注意する。

## 1. 設定

In [ ]:
# セル1：設定エリア  ▼▼▼ 変えてよいのは、ここだけ ▼▼▼
SEED = 42                     # 乱数シード
NOISE_LB = 1e-6               # ノイズ分散の下限（旧ノートは 1e-3）
N_RESTARTS = 5                # GPR の最適化のやり直し回数（増やすと遅いが、安定する）
USE_EXCLUSIONS = True         # True：問題データ一覧の「除外」を使う／False：全条件を使う（取り直した後の確認用）
REPO = ".."                   # リポジトリ直下（このノートは 01_機械学習コードフォルダ にある）
FIG_DPI = 200                 # 保存する PNG の解像度
import numpy as np             # 格子の指定に使う
N_RESTARTS = 3                # 最終モデルの学習でのやり直し回数
MAIN_STRUCT = "B"             # 本命の構造（ユーザー決定、2026-10-12）
ALT_STRUCT = "A"              # 対案の構造
STRUCTS = ["B", "A", "C"]     # 比べる構造（C：構造なし）
H_VARIANTS = ["1", "2", "4", "14", "24"]   # H の扱い：1＝(1) H を残す／2＝(2) m＋c(Fo) に分ける／4＝(4) P3_2（H なし）／14＝(1+4)／24＝(2+4)
DATA_VERSIONS = ["5data", "Alldata"]       # データの版（手順3d）。"5data"：外挿の検証／"Alldata"：全条件で学習し、交差検証
SPLITS = {"5data": ["S_V", "S_T"],         # 版ごとの評価の分け方。5data 版：S_V（外挿）・S_T（210 ℃）・CV（条件単位の交差検証。時間がかかる）
          "Alldata": ["S_T", "CV"]}        # Alldata 版：S_T・CV（10 分割）・S_Vall（5data の条件で学習 → 3data を検証）
MAIN_DATA = "なし"            # 主に使う外れ値の扱い（外れ値なし）
TEMP_HOLDOUT = 210            # S_T でテストにする設定温度 [℃]
E_GRID_A = np.arange(-20.0, 45.01, 2.5)    # A の E_v の粗い格子 [kJ/mol]
EH_GRID = np.arange(-40.0, 40.01, 5.0)     # B の E_h の粗い格子
EV_GRID = np.arange(-20.0, 40.01, 5.0)     # B の E_v の粗い格子
EH_GRID_CV = np.arange(-10.0, 25.01, 5.0)  # CV の各分割での B の粗い格子（時間を抑えるため狭める。最良が端なら ★）
EV_GRID_CV = np.arange(0.0, 35.01, 5.0)
FINE_A = 2.5                  # 細かい格子：A は最良の点の ±FINE_A（0.5 刻み）
FINE_B = 4.0                  # 細かい格子：B は最良の点の ±FINE_B（1 刻み）。CV の分割では FINE_B_CV
FINE_B_CV = 3.0
N_RESTARTS_PROFILE = 3        # 尤度の格子の各点での最適化のやり直し回数
N_RESTARTS_CV = 1             # CV の各分割での学習のやり直し回数（Alldata は行数が多く遅いので少なくする）
B_EH_FIX = None               # None：B の E_h・E_v を両方推定／数（kJ/mol）：E_h をその値に固定（E_v だけ推定）
P32_DRAWS = 20                # (4) で P3_2 を予測して使うときの、予測分布からの引き直しの回数
FIG_VARIANTS = ["1", "2", "4"]  # 曲線などの図に出す H の扱い（全部の指標の表は、すべての扱いで出る）
N_JOBS = 8                    # 並列に計算する数（CPU の数に合わせる）
E_LIT = 40.0                  # 文献値の目安 [kJ/mol]（PP の流動の活性化エネルギー。このデータの「見かけのシフトエネルギー」とは別物）
# ▲▲▲ 設定エリア終了 ▲▲▲

In [ ]:
# セル1.5：ライブラリ、フォント、出力先
%matplotlib inline
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.ticker import FuncFormatter, LogLocator, NullFormatter
from joblib import Parallel, delayed

OUT_DIR = os.path.join(REPO, "results", "step5")
os.makedirs(OUT_DIR, exist_ok=True)
plt.rcParams["figure.dpi"] = 80
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)
try:
    fm.findfont("IPAexGothic", fallback_to_default=False)
except ValueError:
    raise RuntimeError("日本語フォント IPAexGothic が見つかりません。インストールしてからカーネルを再起動してください。")
plt.rcParams["font.family"] = ["IPAexGothic", "DejaVu Sans"]
print("Python:", sys.executable, "/", sys.version.split()[0])
print("出力先:", os.path.abspath(OUT_DIR))

### 共通の部品

データの読み込み、問題データ一覧の「除外」、評価、データの分け方、列を選んで当てるカーネル `OnDims` など。中身は `04_補助スクリプト/nb_common.py` と同じ。

In [ ]:
# セル：共通の部品（読み込み・モデル・評価）。04_補助スクリプト/nb_common.py の中身。直すときは、そのファイルを直して、ノートを作り直す
import os
import re
import time
import warnings

import numpy as np
import pandas as pd
from sklearn.exceptions import ConvergenceWarning
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel as C, DotProduct, Hyperparameter, Kernel, Matern, WhiteKernel
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ---------------------------------------------------------------- 設定（ノートの設定エリアの値があれば、それを使う）
SEED = globals().get("SEED", 42)
NOISE_LB = globals().get("NOISE_LB", 1e-6)                 # ノイズ分散の下限（CLAUDE.md §5。旧ノートは 1e-3）
N_RESTARTS = globals().get("N_RESTARTS", 5)                # GPR の最適化のやり直し回数
REPO = globals().get("REPO", os.environ.get("NB_REPO", ".."))   # リポジトリ直下（ノートは 01_機械学習コードフォルダ にあるので ".."。スクリプトは環境変数 NB_REPO で指定）
USE_EXCLUSIONS = globals().get("USE_EXCLUSIONS", True)     # 問題データ一覧の「除外」を使うか
warnings.filterwarnings("ignore", category=ConvergenceWarning)   # 下限への張り付きは kernels.csv に記録する

# ---------------------------------------------------------------- データの場所
DATA_DIR = os.path.join(REPO, "02_機械学習データ", "NR500")
TEST_DIR = os.path.join(REPO, "03_予測検証用テストデータ", "NR500")
PROBLEM_CSV = os.path.join(REPO, "02_機械学習データ", "問題データ一覧.csv")
MATS = {"PP": "PP", "MIX": "粉砕材混合PP"}
FILES = {   # (材料, 組): (上位フォルダ, 中間フォルダ, フォルダ, ファイル名)。どれも全条件のファイル（問題のある条件は、一覧で除く）
    ("PP", "5data"): (DATA_DIR, "02_バージンPP再々実験_5data", "03_pp_max_5data", "結合_03_pp_max_5data_外れ値{o}.csv"),
    ("PP", "3data"): (TEST_DIR, "02_バージンPP再々実験_5data用検証データ(3data)", "02_pp_max_3data", "結合_02_pp_max_3data_外れ値{o}.csv"),
    ("PP", "Alldata"): (DATA_DIR, "01_バージンPP再々実験_alldata", "01_pp_max_Alldata", "結合_01_pp_max_Alldata_外れ値{o}.csv"),
    ("MIX", "5data"): (DATA_DIR, "04_粉砕材混合PP実験_5data", "03_mixpp_max_5data", "結合_03_mixpp_max_5data_外れ値{o}.csv"),
    ("MIX", "3data"): (TEST_DIR, "04_粉砕材混合PP実験_5data用検証データ(3data)", "02_mixpp_max_3data", "結合_02_mixpp_max_3data_外れ値{o}.csv"),
    ("MIX", "Alldata"): (DATA_DIR, "03_粉砕材混合PP実験_alldata", "01_mixpp_max_Alldata", "結合_01_mixpp_max_Alldata_外れ値{o}.csv"),
}
LEGACY_FILES = {   # 1.5 mm・3 mm/s を除いてあるファイル（no1.5-3.0）。一覧の「除外」で同じデータになるかの確認用
    ("PP", "3data"): (TEST_DIR, "02_バージンPP再々実験_5data用検証データ(3data)", "05_pp_max_3data(no1.5-3.0)", "結合_05_pp_max_3data(no1.5-3.0)_外れ値{o}.csv"),
    ("PP", "Alldata"): (DATA_DIR, "01_バージンPP再々実験_alldata", "04_pp_max_Alldata(no1.5-3.0)", "結合_04_pp_max_Alldata(no1.5-3.0)_外れ値{o}.csv"),
}
COLS = {"元ファイル": "file", "キャビ厚": "H", "射出速度[mm/s]": "v", "樹脂粘度η34(Pa・s)": "eta", "せん断速度γ34(1/s)": "gd",
        "流動中樹脂圧力P3_1(MPa)": "P31", "流動中樹脂圧力P3_2(MPa)": "P32", "加熱シリンダ温度[℃]": "Tset", "流動中樹脂温度(℃)": "Tff"}
RE_SRC = r"^(\d{3})_(\d+)℃_(\d{3})_([\d.]+)mm\.csv$"
D = {}                  # (材料, 組, 外れ値) → DataFrame。ノートで D = load_all() として入れる
REMOVED = {}            # (材料, 組, 外れ値) → 一覧の「除外」で除いた行数


def path_of(mat, grp, o, files=None):
    top, d1, d2, fn = (files or FILES)[(mat, grp)]
    return os.path.join(top, d1, d2, f"外れ値{o}", fn.format(o=o))


# ---------------------------------------------------------------- 問題データ一覧
def read_problem_list(path=None):
    """問題データ一覧.csv を読む（空欄は空文字）"""
    return pd.read_csv(path or PROBLEM_CSV, encoding="utf-8-sig", dtype=str).fillna("")


def problem_mask(df, mat, grp, plist, kinds):
    """区分が kinds のどれかで、再測定が「済み」でない一覧の行のうち、この材料・組・条件に当てはまる行の真偽値と、その ID を返す"""
    mask = np.zeros(len(df), bool)
    ids = [[] for _ in range(len(df))]
    for _, r in plist.iterrows():
        if r["区分"] not in kinds or r["再測定"] == "済み" or r["データ"] not in ("新", ""):
            continue
        if r["材料"] not in ("すべて", "", MATS[mat]) or r["組"] not in ("", grp):
            continue
        m = np.ones(len(df), bool)
        if r["設定温度"] != "":
            m &= df["Tset"].to_numpy() == float(r["設定温度"])
        if r["厚み"] != "":
            m &= np.isclose(df["H"].to_numpy(), float(r["厚み"]))
        if r["射出速度"] != "":
            m &= df["v"].to_numpy() == float(r["射出速度"])
        if r.get("ショット", "") != "":                      # 空欄は条件全体。数字なら、そのショット（元ファイル名の先頭 3 桁）だけ
            m &= (df["shot"].to_numpy() == int(r["ショット"])) if "shot" in df.columns else np.zeros(len(df), bool)
        mask |= m
        for i in np.flatnonzero(m):
            ids[i].append(r["ID"])
    return mask, [";".join(x) for x in ids]


def first_shots(mat, grp):
    """条件ごとの、最初のショットの番号（外れ値ありのファイル。ショットが除かれる前の番号）。ショットの順番 pos の基準に使う"""
    ref = pd.read_csv(path_of(mat, grp, "あり"), encoding="utf-8-sig").rename(columns=COLS)
    ref["shot0"] = ref["file"].str.extract(RE_SRC)[0].astype(int)
    ref["Tset"] = ref["Tset"].astype(float)
    return ref.groupby(["Tset", "H", "v"], as_index=False)["shot0"].min()


def load(mat, grp, o, plist=None, exclude=None):
    """結合ファイルを読み、形式を確かめ、問題データ一覧の「除外」を除く。列 sens には、「感度確認」の ID が入る（なければ空文字）。
    列 shot は元ファイル名の先頭 3 桁（設定温度・厚みごとの通し番号で、同じ条件のショットは連続した番号）。列 pos は、その条件の中での順番（最初のショット = 1。外れ値で除かれたショットの番号は欠ける）"""
    exclude = USE_EXCLUSIONS if exclude is None else exclude
    df = pd.read_csv(path_of(mat, grp, o), encoding="utf-8-sig")
    assert list(df.columns) == list(COLS), f"列名が想定と違う: {list(df.columns)}"
    df = df.rename(columns=COLS)
    assert df.isna().sum().sum() == 0, "欠損がある"
    p = df["file"].str.extract(RE_SRC)
    assert p.notna().all().all(), "ファイル名の形式に合わない行がある"
    assert np.allclose(p[1].astype(float), df["Tset"]) and np.allclose(p[2].astype(float), df["v"]) \
        and np.allclose(p[3].astype(float), df["H"]), "ファイル名と列の値が一致しない"
    df["shot"] = p[0].astype(int)
    df["Tset"] = df["Tset"].astype(float)
    df = df.merge(first_shots(mat, grp), on=["Tset", "H", "v"], how="left")
    assert df["shot0"].notna().all(), "外れ値ありのファイルにない条件がある"
    df["pos"] = df["shot"] - df["shot0"].astype(int) + 1
    df = df.drop(columns="shot0")
    plist = read_problem_list() if plist is None else plist
    n0 = len(df)
    if exclude:
        drop, _ = problem_mask(df, mat, grp, plist, ("除外",))
        df = df[~drop].reset_index(drop=True)
    REMOVED[(mat, grp, o)] = n0 - len(df)
    _, df["sens"] = problem_mask(df, mat, grp, plist, ("感度確認",))
    df["is_sens"] = df["sens"] != ""
    df["cond"] = df["Tset"].astype(int).astype(str) + "_" + df["H"].astype(str) + "_" + df["v"].astype(int).astype(str)
    df["ln_eta"], df["ln_gd"], df["ln_H"] = np.log(df["eta"]), np.log(df["gd"]), np.log(df["H"])
    df["ln_P31"], df["ln_P32"] = np.log(df["P31"] * 1e6), np.log(df["P32"] * 1e6)   # CSV の圧力は MPa。Pa に直してから ln をとる
    df["ln_v"] = np.log(df["v"])
    df["ln_Tff"] = np.log(df["Tff"] + 273.15)          # 流動先端温度 [K]
    for c in ["gd", "Tff", "P32"]:   # 条件ごとの平均（成形条件から見たばらつきを見るモデル用。cm は condition mean）
        df[f"ln_{c}_cm"] = df.groupby("cond")[f"ln_{c}"].transform("mean")
    return df


def load_all(exclude=None):
    """全材料・全組・外れ値なし/あり を読む。学習と検証で条件が重ならないことも確かめる"""
    plist = read_problem_list()
    out = {(mat, grp, o): load(mat, grp, o, plist, exclude) for mat in MATS for grp in ["5data", "3data", "Alldata"] for o in ["なし", "あり"]}
    for mat in MATS:
        for o in ["なし", "あり"]:
            assert not (set(out[(mat, "5data", o)]["cond"]) & set(out[(mat, "3data", o)]["cond"])), "学習と検証で条件が重なっている"
    return out


def legacy_check(D_):
    """全条件のファイルから一覧で除いたデータが、1.5 mm・3 mm/s を除いてあるファイル（no1.5-3.0）と同じ行か（PP の 3data・Alldata）。
    no1.5-3.0 のファイルにも、1.5 mm・3 mm/s 以外の一覧の「除外」（P03 など）を同じように適用してから比べる"""
    rows = []
    plist = read_problem_list()
    plist = plist[~((plist["厚み"] == "1.5") & (plist["射出速度"] == "3") & (plist["設定温度"] == ""))]      # no1.5-3.0 のファイルで、すでに除かれている分
    for (mat, grp), _ in LEGACY_FILES.items():
        for o in ["なし", "あり"]:
            old = pd.read_csv(path_of(mat, grp, o, LEGACY_FILES), encoding="utf-8-sig")
            o2 = old.rename(columns=COLS)
            o2["shot"] = o2["file"].str.extract(RE_SRC)[0].astype(int)
            o2["Tset"] = o2["Tset"].astype(float)
            drop, _ = problem_mask(o2, mat, grp, plist, ("除外",)) if USE_EXCLUSIONS else (np.zeros(len(o2), bool), None)
            old = old[~drop]
            same = set(old["元ファイル"]) == set(D_[(mat, grp, o)]["file"])
            rows.append({"材料": MATS[mat], "組": grp, "外れ値": o, "一覧で除いた後の行数": len(D_[(mat, grp, o)]), "no1.5-3.0 の行数": len(old), "同じ行": same})
    return pd.DataFrame(rows)


# ---------------------------------------------------------------- モデル
def kernel_for(d, linear=False):
    """ARD Matern 5/2 ＋ ノイズ。linear=True なら、線形カーネル（DotProduct）も足す（低速側への外挿のため。CLAUDE.md §5）"""
    k = C(1.0, (1e-3, 1e3)) * Matern(length_scale=[1.0] * d, length_scale_bounds=(1e-2, 1e2), nu=2.5)
    if linear:
        k = k + C(1.0, (1e-3, 1e3)) * DotProduct(sigma_0=1.0, sigma_0_bounds=(1e-5, 1e3))
    return k + WhiteKernel(noise_level=0.1, noise_level_bounds=(NOISE_LB, 1e1))


class OnDims(Kernel):
    """入力の一部の列 dims にだけ当てるカーネル（足し算のカーネル m(…) + c(…) を作るための部品。手順5）。
    ハイパーパラメータは、中のカーネルのものをそのまま使う（scikit-learn の Exponentiation と同じ作り）。tag は、どの部品かの印（"m"・"c" など）"""

    def __init__(self, kernel, dims, tag=""):
        self.kernel = kernel
        self.dims = dims                                   # 列番号のリスト（scikit-learn の clone の決まりで、受け取ったものをそのまま持つ）
        self.tag = tag

    def get_params(self, deep=True):
        params = dict(kernel=self.kernel, dims=self.dims, tag=self.tag)
        if deep:
            params.update(("kernel__" + k, val) for k, val in self.kernel.get_params().items())
        return params

    @property
    def hyperparameters(self):
        return [Hyperparameter("kernel__" + h.name, h.value_type, h.bounds, h.n_elements) for h in self.kernel.hyperparameters]

    @property
    def theta(self):
        return self.kernel.theta

    @theta.setter
    def theta(self, theta):
        self.kernel.theta = theta

    @property
    def bounds(self):
        return self.kernel.bounds

    def __eq__(self, b):
        return type(self) == type(b) and self.dims == b.dims and self.kernel == b.kernel

    def __hash__(self):
        return hash((tuple(self.dims), self.tag))

    def __call__(self, X, Y=None, eval_gradient=False):
        Xd = np.asarray(X)[:, self.dims]
        Yd = None if Y is None else np.asarray(Y)[:, self.dims]
        return self.kernel(Xd, Yd, eval_gradient=eval_gradient)

    def diag(self, X):
        return self.kernel.diag(np.asarray(X)[:, self.dims])

    def is_stationary(self):
        return self.kernel.is_stationary()

    def __repr__(self):
        return f"OnDims({self.tag}, dims={self.dims}, {self.kernel!r})"


def make_gpr(d, linear=False, restarts=None):
    return GaussianProcessRegressor(kernel=kernel_for(d, linear), n_restarts_optimizer=N_RESTARTS if restarts is None else restarts, random_state=SEED)


def fit_ols(X_tr, y_tr):
    """切片ありの最小二乗回帰。係数、(XᵀX)⁻¹、残差分散を返す"""
    A_tr = np.column_stack([np.ones(len(X_tr)), X_tr])
    beta, *_ = np.linalg.lstsq(A_tr, y_tr, rcond=None)
    resid = y_tr - A_tr @ beta
    return beta, np.linalg.inv(A_tr.T @ A_tr), resid @ resid / (len(y_tr) - A_tr.shape[1])


def predict_ols(model, X):
    """予測平均と、新しい1ショットに対する予測標準偏差"""
    beta, XtX_inv, s2 = model
    A_new = np.column_stack([np.ones(len(X)), X])
    lev = np.einsum("ij,jk,ik->i", A_new, XtX_inv, A_new)
    return A_new @ beta, np.sqrt(s2 * (1.0 + lev))


MODELS = {}             # 記号 → {"kind": "ols" か "gpr", "cols": [入力の列], "linear": 線形カーネルを足すか}。ノートで決める


def fit_model(key, tr, restarts=None):
    spec = MODELS[key]
    X, y = tr[spec["cols"]].to_numpy(), tr["ln_eta"].to_numpy()
    if spec["kind"] == "ols":
        return {"kind": "ols", "model": fit_ols(X, y)}
    sc = StandardScaler().fit(X)              # 標準化と中心化は学習データだけで計算する
    y0 = y.mean()
    gp = make_gpr(X.shape[1], spec.get("linear", False), restarts).fit(sc.transform(X), y - y0)
    return {"kind": "gpr", "gp": gp, "sc": sc, "y0": y0}


def predict_model(key, fit, te):
    """予測平均、予測標準偏差（ノイズ込み）、潜在関数の標準偏差、ノイズの標準偏差"""
    X = te[MODELS[key]["cols"]].to_numpy()
    if fit["kind"] == "ols":
        mu, sd = predict_ols(fit["model"], X)
        return mu, sd, np.full(len(X), np.nan), float(np.sqrt(fit["model"][2]))
    mu, sd = fit["gp"].predict(fit["sc"].transform(X), return_std=True)
    noise = fit["gp"].kernel_.k2.noise_level
    return mu + fit["y0"], sd, np.sqrt(np.maximum(sd ** 2 - noise, 0.0)), float(np.sqrt(noise))


def evaluate(y, mu, sd):
    """ln η の指標：R2、RMSE、95% 予測区間の被覆率、NLPD（各ショットの予測分布の負の対数密度の平均）、予測標準偏差の平均"""
    z = (y - mu) / sd
    return {"R2": r2_score(y, mu) if len(y) > 1 else np.nan, "RMSE": float(np.sqrt(mean_squared_error(y, mu))),
            "coverage95": float(np.mean(np.abs(z) <= 1.96)),
            "NLPD": float(np.mean(0.5 * np.log(2 * np.pi * sd ** 2) + 0.5 * z ** 2)), "mean_sd": float(np.mean(sd))}


def kernel_row(key, fit):
    """学習した GPR のハイパーパラメータ（振幅、長さスケール、ノイズ、線形項、対数周辺尤度、下限・上限への張り付き）"""
    gp = fit["gp"]
    k = gp.kernel_
    lin = MODELS[key].get("linear", False)
    prod, lin_prod = (k.k1.k1, k.k1.k2) if lin else (k.k1, None)
    ls = np.atleast_1d(prod.k2.length_scale)
    row = {"amplitude": float(np.sqrt(prod.k1.constant_value)), "noise_var": float(k.k2.noise_level),
           "noise_sd": float(np.sqrt(k.k2.noise_level)), "LML": float(gp.log_marginal_likelihood_value_),
           "noise_at_lower_bound": bool(k.k2.noise_level < NOISE_LB * 1.05),
           "ls_at_bound": bool(np.any(ls > 99) or np.any(ls < 0.0105))}
    if lin_prod is not None:
        row.update({"lin_amplitude": float(np.sqrt(lin_prod.k1.constant_value)), "lin_sigma0": float(lin_prod.k2.sigma_0)})
    row.update({f"ls_{c}": float(l) for c, l in zip(MODELS[key]["cols"], ls)})
    return row


def within_stats(df):
    """同じ条件内のばらつき（プール）。dof：自由度、var_after_gd：ln γ̇ で説明した残りの分散"""
    g = df.groupby("cond")
    dof = int((g.size() - 1).sum())
    de = df["ln_eta"] - g["ln_eta"].transform("mean")
    dg = df["ln_gd"] - g["ln_gd"].transform("mean")
    var_e = float((de ** 2).sum() / dof)
    b = float((de * dg).sum() / (dg ** 2).sum())
    return {"dof": dof, "var_ln_eta": var_e, "sd_ln_eta": var_e ** 0.5, "corr_with_ln_gd": float((de * dg).sum() / np.sqrt((de ** 2).sum() * (dg ** 2).sum())),
            "slope_vs_ln_gd": b, "var_after_gd": float(((de - b * dg) ** 2).sum() / (dof - 1))}


# ---------------------------------------------------------------- データの分け方
def split_V(mat, o):
    """5data で学習し、3data で検証する（射出速度の両端の外挿）。te の group 列は、低速側（3・5 mm/s）か高速側（280・320 mm/s）"""
    tr, te = D[(mat, "5data", o)].copy(), D[(mat, "3data", o)].copy()
    te["group"] = np.where(te["v"] <= 5, "低速側", np.where(te["v"] >= 280, "高速側", "その他"))
    assert (te["group"] != "その他").all(), "検証データに、低速側・高速側のどちらでもない条件がある"
    return tr, te


def split_random(df):
    """行単位のランダム 20%（従来の分け方。random_state = SEED）"""
    idx_tr, idx_te = train_test_split(np.arange(len(df)), test_size=0.2, random_state=SEED)
    te = df.iloc[idx_te].copy()
    te["group"] = "全体"
    return df.iloc[idx_tr].copy(), te


def split_T(mat, o, t_test=210):
    """設定温度 t_test の全条件をテストにし、5data の残りの温度で学習する（温度の内挿の検証）。3data の同じ温度は te3 として別に返す"""
    d5, d3 = D[(mat, "5data", o)], D[(mat, "3data", o)]
    tr, te = d5[d5["Tset"] != t_test].copy(), d5[d5["Tset"] == t_test].copy()
    te["group"] = "全体"
    te3 = d3[d3["Tset"] == t_test].copy()
    te3["group"] = np.where(te3["v"] <= 5, "低速側", "高速側")
    return tr, te, te3


def pool_of(mat, o):
    """5data と 3data をまとめたデータ"""
    return pd.concat([D[(mat, "5data", o)], D[(mat, "3data", o)]], ignore_index=True)


# ---------------------------------------------------------------- 学習と評価の記録
METRICS, KERNELS, OLSROWS, PRED, FITS = [], [], [], {}, {}      # FITS：(材料, split, データ, モデル) → 学習した予測器
SENS_SUFFIX = "（感度確認の条件を除く）"


def _record(mat, name, data, key, fit, tr, te, pure):
    """学習した予測器 fit で te を予測し、指標（subset ごと）と予測を記録する"""
    subsets = {"全体": np.ones(len(te), bool)}
    for g in te["group"].unique():
        if g != "全体":
            subsets[g] = (te["group"] == g).to_numpy()
    if te["is_sens"].any():
        subsets.update({s + SENS_SUFFIX: m & ~te["is_sens"].to_numpy() for s, m in list(subsets.items())})
    y_te = te["ln_eta"].to_numpy()
    mu, sd, sd_f, noise_sd = predict_model(key, fit, te)
    PRED[(mat, name, data, key)] = pd.DataFrame({"y": y_te, "mu": mu, "sd": sd, "sd_f": sd_f, "v": te["v"].to_numpy(), "H": te["H"].to_numpy(),
                                                 "Tset": te["Tset"].to_numpy(), "cond": te["cond"].to_numpy(), "group": te["group"].to_numpy(),
                                                 "is_sens": te["is_sens"].to_numpy()})
    for sub, mask in subsets.items():
        if mask.sum() == 0:
            continue
        ev = evaluate(y_te[mask], mu[mask], sd[mask])
        METRICS.append({"material": MATS[mat], "split": name, "data": data, "model": key, "subset": sub, "n_train": len(tr),
                        "n_test": int(mask.sum()), "n_cond_test": int(te.loc[mask, "cond"].nunique()), **ev,
                        "mean_sd_latent": float(np.nanmean(sd_f[mask])) if fit["kind"] == "gpr" else np.nan, "noise_sd": noise_sd})


def run_split(mat, split, data, tr, te, keys=None, tag="", extra_te=None):
    """全モデルを学習し、指標（subset ごと）・カーネル・予測を記録する。te には group 列がある。
    te に「感度確認」の条件があれば、各 subset から除いた指標（subset 名の後ろに SENS_SUFFIX）も記録する。
    tag は split 名の後ろに付ける印。extra_te（{split 名: テスト}）があれば、同じ学習で、それらも評価する"""
    keys = keys or ALL_KEYS
    name = split + tag
    tests = {name: te, **{s + tag: t for s, t in (extra_te or {}).items()}}
    pure = within_stats(tr)
    for key in keys:
        t0 = time.time()
        fit = fit_model(key, tr)
        FITS[(mat, name, data, key)] = fit
        for nm, te_ in tests.items():
            _record(mat, nm, data, key, fit, tr, te_, pure)
        base = {"material": MATS[mat], "split": name, "data": data, "model": key, "n_train": len(tr)}
        if fit["kind"] == "gpr":
            KERNELS.append({**base, **kernel_row(key, fit), "pure_sd_ln_eta": pure["sd_ln_eta"], "pure_sd_after_gd": pure["var_after_gd"] ** 0.5})
        else:
            names = ["intercept"] + MODELS[key]["cols"]
            OLSROWS.append({**base, "resid_sd": float(np.sqrt(fit["model"][2])), **dict(zip(names, map(float, fit["model"][0])))})
        print(f"{MATS[mat]} {name} {data} {key}: 学習 {len(tr)} 行、{time.time() - t0:.1f} 秒")


# ---------------------------------------------------------------- 条件を1つずつ抜く交差検証（LOCO）
def loco_gpr(key, pool):
    """GPR：ハイパーパラメータを全データで決めて固定し、条件を抜いた予測を閉じた式で計算する（平均 = y_g − [K⁻¹_gg]⁻¹(K⁻¹y)_g、共分散 = [K⁻¹_gg]⁻¹）"""
    spec = MODELS[key]
    X, y = pool[spec["cols"]].to_numpy(), pool["ln_eta"].to_numpy()
    sc = StandardScaler().fit(X)
    Xs, yc = sc.transform(X), y - y.mean()
    gp = make_gpr(X.shape[1], spec.get("linear", False)).fit(Xs, yc)
    Kinv = np.linalg.inv(gp.kernel_(Xs))              # ノイズ込みの共分散行列の逆行列
    alpha = Kinv @ yc
    mu, sd = np.empty(len(y)), np.empty(len(y))
    for idx in pool.groupby("cond").indices.values():
        Minv = np.linalg.inv(Kinv[np.ix_(idx, idx)])
        mu[idx] = y[idx] - Minv @ alpha[idx]
        sd[idx] = np.sqrt(np.diag(Minv))
    return mu, sd, gp


def loco_ols(key, pool):
    """線形回帰：条件ごとに、その条件を除いて当てはめ直す"""
    cols = MODELS[key]["cols"]
    X, y = pool[cols].to_numpy(), pool["ln_eta"].to_numpy()
    mu, sd = np.empty(len(y)), np.empty(len(y))
    for idx in pool.groupby("cond").indices.values():
        keep = np.ones(len(y), bool)
        keep[idx] = False
        mu[idx], sd[idx] = predict_ols(fit_ols(X[keep], y[keep]), X[idx])
    return mu, sd


# ---------------------------------------------------------------- 外れ値の検知（規則 f）と、データの版（5data 版・Alldata 版）
OUTLIER_Z = globals().get("OUTLIER_Z", 3.5)             # 規則 f：|z| がこれを超え、かつ |残差|（ln η）が次の値を超える条件を候補にする
OUTLIER_RESID = globals().get("OUTLIER_RESID", 0.1)     # 0.1 は粘度で約 10%


def cond_means(df):
    """条件ごとの平均（ln η と ln γ̇ をショットで平均した点）"""
    g = df.groupby("cond", sort=False)
    t = g.agg(Tset=("Tset", "first"), H=("H", "first"), v=("v", "first"), ln_eta=("ln_eta", "mean"), ln_gd=("ln_gd", "mean"), n=("ln_eta", "size")).reset_index()
    t["ln_H"] = np.log(t["H"])
    return t


def outlier_scan(df, z_thr=None, resid_thr=None):
    """規則 f：条件平均が、ほかの条件から予測した値と大きく違う条件を拾う。
    条件平均の GPR（入力 ln γ̇（条件平均）・ln H・Tset。R3 と同じ形）の、条件を 1 つずつ抜いた予測（閉じた式）との残差を、予測標準偏差で割って z とする。
    返す表の列：cond, Tset, H, v, n（ショット数）, resid（ln η の実測 − 予測）, sd, z, flag（|z| と |resid| が両方しきい値を超えたか）"""
    z_thr = OUTLIER_Z if z_thr is None else z_thr
    resid_thr = OUTLIER_RESID if resid_thr is None else resid_thr
    t = cond_means(df)
    X, y = t[["ln_gd", "ln_H", "Tset"]].to_numpy(), t["ln_eta"].to_numpy()
    Xs, yc = StandardScaler().fit_transform(X), y - y.mean()
    gp = make_gpr(3, True).fit(Xs, yc)
    Kinv = np.linalg.inv(gp.kernel_(Xs))
    d = np.diag(Kinv)
    t["resid"], t["sd"] = (Kinv @ yc) / d, np.sqrt(1.0 / d)           # 条件 i を抜いた予測：平均 = y_i − (K⁻¹y)_i / K⁻¹_ii、分散 = 1 / K⁻¹_ii
    t["z"] = t["resid"] / t["sd"]
    t["flag"] = (t["z"].abs() > z_thr) & (t["resid"].abs() > resid_thr)
    return t[["cond", "Tset", "H", "v", "n", "resid", "sd", "z", "flag"]]


DATA_VERSIONS = globals().get("DATA_VERSIONS", ["5data", "Alldata"])   # 検討するデータの版。"5data"：5data で学習し、3data で外挿を検証する版／"Alldata"：全条件で学習する版
CV_K = globals().get("CV_K", 10)                                       # Alldata 版の交差検証の分割数（条件単位）


def split_cv(pool, k=None):
    """条件単位の k 分割交差検証。同じ条件のショットは同じ側に入れる（SEED で条件を並べ替える）。(学習, テスト) の組の一覧を返す"""
    k = CV_K if k is None else k
    conds = np.array(sorted(pool["cond"].unique()))
    np.random.RandomState(SEED).shuffle(conds)
    out = []
    for i in range(k):
        te_c = set(conds[i::k])
        te = pool[pool["cond"].isin(te_c)].copy()
        te["group"] = "全体"
        out.append((pool[~pool["cond"].isin(te_c)].copy(), te))
    return out


def version_sets(mat, version, data="なし", t_test=210):
    """データの版ごとの分割を返す。{"pool": 最終モデル（ALL）の学習データ, "splits": {分割の名前: (学習, {集計の名前: テスト})}}
    5data 版：S_V（5data → 3data。低速側・高速側）、S_T（190・230 ℃ → 210 ℃。5data の 210 ℃ と 3data の 210 ℃ を別に集計）、CV（5data の中の条件単位の CV_K 分割）。最終モデルは 5data（3data は検証専用）。
    Alldata 版：S_T（Alldata の 190・230 ℃ で学習。テストは 5data 版と同じ行：5data ファイルの 210 ℃ と 3data ファイルの 210 ℃。そのまま比べられる）、
    S_Vall（Alldata のうち 5data の条件だけで学習 → 3data の条件を検証。PP のみ意味がある〔粉砕材混合 PP は 5data と同じ〕）、CV（Alldata の条件単位の CV_K 分割）。最終モデルは Alldata（全条件）"""
    d5, d3 = D[(mat, "5data", data)], D[(mat, "3data", data)]
    if version == "5data":
        tr_v, te_v = split_V(mat, data)
        tr_t, te_t, te_t3 = split_T(mat, data, t_test)
        sp = {"S_V": (tr_v, {"S_V": te_v}), "S_T": (tr_t, {"S_T": te_t, "S_T3": te_t3})}
        pool = d5
    else:
        assert version == "Alldata", version
        pool = D[(mat, "Alldata", data)]
        _, te_t, te_t3 = split_T(mat, data, t_test)
        sp = {"S_T": (pool[pool["Tset"] != t_test].copy(), {"S_T": te_t, "S_T3": te_t3})}
        te_v = d3.copy()
        te_v["group"] = np.where(te_v["v"] <= 5, "低速側", np.where(te_v["v"] >= 280, "高速側", "その他"))
        sp["S_Vall"] = (pool[pool["cond"].isin(set(d5["cond"]))].copy(), {"S_Vall": te_v})
    for i, (tr_i, te_i) in enumerate(split_cv(pool), 1):
        sp[f"CV{i}"] = (tr_i, {f"CV{i}": te_i})
    return {"pool": pool, "splits": sp}


# ---------------------------------------------------------------- 判定の表示（GPR6月ノートの「判定」にならう）
def judge(ok, text):
    """結果の判定を1行で表示する。ok が True なら「OK」、False なら「要確認」"""
    print(f">> 判定：{'OK' if ok else '★要確認'}　{text}")


# ---------------------------------------------------------------- 高解像度のグラフ（GPR6月ノートのセル9-2 にならう。報告書・発表資料用）
HR_COLORS = {"H": {0.5: "#d62728", 1.0: "#1f77b4", 1.5: "#f0b400", 2.0: "#2ca02c"},       # 厚みごとの色（6月のノートと同じ並び。黄は、線で見やすい濃さにした）
             "Tset": {190: "#1f77b4", 210: "#2ca02c", 230: "#d62728"}}                       # 設定温度ごとの色
HR_MARKS = {"H": {0.5: "o", 1.0: "s", 1.5: "^", 2.0: "D"}, "Tset": {190: "o", 210: "s", 230: "^"}}
HR_DEFAULT = dict(layouts=["thickness", "temperature"], band="predict", points="cond", show_test=True, mark_sens=True, x_range=None, extend=1.3,
                  xlim={"thickness": None, "temperature": (30, 50000)}, ylim={"thickness": None, "temperature": (0.5, 5000)}, figsize=(15, 10), dpi=600, font={"axis": 24, "legend": 24, "title": 24, "tick": 28},
                  marker=200, marker_shot=70, line=3.0, show_title=True, show_xlabel=False, show_ylabel=False, formats=["png"], show=False, n_grid=120)


def hr_split_conds(mat, data, split, t_test=210):
    """学習に使った条件の集合。S_V：5data／S_T：5data の t_test 以外の温度／ALL：5data と 3data"""
    d5, d3 = D[(mat, "5data", data)], D[(mat, "3data", data)]
    if split == "S_V":
        return set(d5["cond"])
    if split == "S_T":
        return set(d5.loc[d5["Tset"] != t_test, "cond"])
    return set(d5["cond"]) | set(d3["cond"])


def hr_inputs(sub, lg, h, t):
    """予測線の入力（ln γ̇ = lg の点）。Tff・P3_2・射出速度は、同じ設定温度・厚みの条件平均を ln γ̇ で内挿する（どの入力のモデルでも描けるように）"""
    cm = sub.groupby("cond").agg(lg=("ln_gd", "mean"), tff=("ln_Tff", "mean"), p32=("ln_P32", "mean"), lv=("ln_v", "mean")).sort_values("lg")
    ip = lambda c: np.interp(lg, cm["lg"], cm[c])
    return pd.DataFrame({"ln_gd": lg, "ln_gd_cm": lg, "ln_H": np.log(h), "Tset": float(t), "ln_Tff": ip("tff"), "ln_Tff_cm": ip("tff"),
                         "ln_P32": ip("p32"), "ln_P32_cm": ip("p32"), "ln_v": ip("lv")})


HR_ALL = globals().get("HR_ALL", True)                 # すべての図を、高解像度でも保存するか（True：out_dir/highres/ に HR_ALL_DPI で保存。Git の管理外）
HR_ALL_DPI = globals().get("HR_ALL_DPI", 600)


def save_fig(fig, out_dir, stem):
    """図を out_dir に PNG で保存する（解像度は FIG_DPI）。HR_ALL が True なら、同じ図を HR_ALL_DPI（既定 600 dpi）で out_dir/highres/ にも保存する。保存したファイルの一覧を返す"""
    os.makedirs(out_dir, exist_ok=True)
    paths = [os.path.join(out_dir, f"{stem}.png")]
    fig.savefig(paths[0], dpi=globals().get("FIG_DPI", 200), bbox_inches="tight")
    if HR_ALL:
        os.makedirs(os.path.join(out_dir, "highres"), exist_ok=True)
        paths.append(os.path.join(out_dir, "highres", f"{stem}.png"))
        fig.savefig(paths[1], dpi=HR_ALL_DPI, bbox_inches="tight")
    return paths


def hr_flow(predict, mat, data, train_conds, title_model, stem, out_dir, pool=None, **kw):
    """せん断速度 ― 粘度の学習結果グラフを、高解像度で 1 枚ずつ保存する（6月のノートのセル9-2 と同じ見せ方）。
    predict(Xg) → (予測平均 ln η, 予測標準偏差（ノイズ込み）, 潜在関数の標準偏差)。Xg は hr_inputs の DataFrame。
    train_conds：学習に使った条件の集合（それ以外は、検証の点として白抜きで重ねる）。pool：点と曲線の範囲に使うデータ（None なら 5data＋3data）。
    設定は HR_DEFAULT を参照（kw で上書き）。保存したファイルの一覧を返す"""
    import matplotlib.pyplot as plt
    import matplotlib.ticker as ticker
    from matplotlib.lines import Line2D
    from matplotlib.patches import Patch
    cfg = {**HR_DEFAULT, **kw}
    os.makedirs(out_dir, exist_ok=True)
    pool = pool_of(mat, data) if pool is None else pool
    curves = {}                                                         # (厚み, 設定温度) → (x, 予測平均, 帯の幅の標準偏差)。レイアウトが違っても同じ線なので、1 回だけ計算する
    for (h, t), sub in pool.groupby(["H", "Tset"]):
        lgc = sub.groupby("cond")["ln_gd"].mean()
        if len(lgc) < 2:
            continue
        lo, hi = (np.log(cfg["x_range"][0]), np.log(cfg["x_range"][1])) if cfg["x_range"] else (lgc.min() - np.log(cfg["extend"]), lgc.max() + np.log(cfg["extend"]))
        lg = np.linspace(lo, hi, cfg["n_grid"])
        mu, sd, sdf = predict(hr_inputs(sub, lg, h, t))
        curves[(float(h), int(t))] = (np.exp(lg), np.asarray(mu), np.asarray(sd if cfg["band"] == "predict" else sdf))
    saved = []
    ft = cfg["font"]
    band_name = "95% 予測区間" if cfg["band"] == "predict" else "95% 信頼区間"
    for layout in cfg["layouts"]:
        by, sk = ("H", "Tset") if layout == "thickness" else ("Tset", "H")        # by：1 枚ごとに固定する量、sk：1 枚の中で線を分ける量
        for gv in sorted({k[0] if by == "H" else k[1] for k in curves}):
            fig, ax = plt.subplots(figsize=cfg["figsize"], dpi=cfg["dpi"])
            series, any_test, any_sens, xs, ys = [], False, False, [], []
            for sv in sorted({(k[1] if by == "H" else k[0]) for k in curves if (k[0] if by == "H" else k[1]) == gv}):
                h, t = (gv, sv) if by == "H" else (sv, gv)
                col, mk = HR_COLORS[sk][sv], HR_MARKS[sk][sv]
                x, mu, s = curves[(float(h), int(t))]
                if cfg["band"] in ("predict", "latent"):
                    ax.fill_between(x, np.exp(mu - 1.96 * s), np.exp(mu + 1.96 * s), color=col, alpha=0.22, zorder=15)
                    ax.plot(x, np.exp(mu - 1.96 * s), color=col, lw=0.6, alpha=0.7, zorder=16)
                    ax.plot(x, np.exp(mu + 1.96 * s), color=col, lw=0.6, alpha=0.7, zorder=16)
                ax.plot(x, np.exp(mu), color=col, lw=cfg["line"], zorder=20)
                sub = pool[(np.isclose(pool["H"], h)) & (pool["Tset"] == t)]
                if cfg["points"] == "cond":
                    a = sub.groupby("cond").agg(lg=("ln_gd", "mean"), le=("ln_eta", "mean"), sens=("is_sens", "max")).reset_index()
                    px, py, tr_m, se_m, msize = np.exp(a["lg"].to_numpy()), np.exp(a["le"].to_numpy()), a["cond"].isin(train_conds).to_numpy(), a["sens"].to_numpy().astype(bool), cfg["marker"]
                else:
                    px, py, tr_m, se_m, msize = sub["gd"].to_numpy(), sub["eta"].to_numpy(), sub["cond"].isin(train_conds).to_numpy(), sub["is_sens"].to_numpy().astype(bool), cfg["marker_shot"]
                xs += list(x) + list(px)
                ys += list(np.exp(mu)) + list(py)
                ax.scatter(px[tr_m], py[tr_m], s=msize, color=col, marker=mk, alpha=0.85, edgecolors="k", linewidths=0.6, zorder=14 - list(HR_COLORS[sk]).index(sv))
                if cfg["show_test"] and (~tr_m).any():
                    ax.scatter(px[~tr_m], py[~tr_m], s=msize, facecolors="white", edgecolors=col, marker=mk, linewidths=2.5, zorder=14 - list(HR_COLORS[sk]).index(sv))
                    any_test = True
                if cfg["mark_sens"] and se_m.any():
                    ax.scatter(px[se_m], py[se_m], s=msize * 1.3, color="k", marker="x", linewidths=3, zorder=25)
                    any_sens = True
                series.append((sv, col, mk))
            head = "キャビティ厚み" if sk == "H" else "設定温度"
            handles = [Patch(facecolor="gray", alpha=0.3, label=band_name), Patch(color="none", label=head)]
            handles += [Line2D([], [], color=col, lw=cfg["line"], marker=mk, markersize=np.sqrt(cfg["marker"]), markeredgecolor="k", markeredgewidth=0.6,
                               label=(f"{sv} mm" if sk == "H" else f"{int(sv)} ℃")) for sv, col, mk in series]
            if any_test:
                handles.append(Line2D([], [], color="gray", ls="none", marker="o", markersize=np.sqrt(cfg["marker"]), markerfacecolor="white", markeredgewidth=2.5, label="検証の条件（白抜き）"))
            if any_sens:
                handles.append(Line2D([], [], color="k", ls="none", marker="x", markersize=np.sqrt(cfg["marker"]), markeredgewidth=3, label="感度確認の条件"))
            if cfg["band"] not in ("predict", "latent"):
                handles = handles[1:]
            ax.legend(handles=handles, loc="upper right", fontsize=ft["legend"], framealpha=1.0, edgecolor="black")
            ax.tick_params(axis="both", which="both", direction="in", top=True, right=True, labelsize=ft["tick"], pad=8)
            ax.set_xscale("log")
            ax.set_yscale("log")
            xl = cfg["xlim"][layout] if isinstance(cfg["xlim"], dict) else cfg["xlim"]       # 範囲：(下限, 上限)＝固定／None＝その図のデータに合わせる
            yl = cfg["ylim"][layout] if isinstance(cfg["ylim"], dict) else cfg["ylim"]
            auto_x, auto_y = xl is None, yl is None
            ax.set_xlim(*((min(xs) / 1.4, max(xs) * 1.4) if auto_x else xl))
            ax.set_ylim(*((min(ys) / 1.5, max(ys) * 1.5) if auto_y else yl))
            for axis, auto in ((ax.xaxis, auto_x), (ax.yaxis, auto_y)):
                if auto:                               # 範囲が狭いので、1・2・5 × 10^n に目盛りを付けて、普通の数で表す
                    axis.set_major_locator(ticker.LogLocator(base=10.0, subs=(1.0, 2.0, 5.0)))
                    axis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f"{v:g}"))
                else:                                  # 6月のノートと同じ（10 の累乗の目盛り）
                    axis.set_major_locator(ticker.LogLocator(base=10.0, subs=(1.0,)))
                    axis.set_major_formatter(ticker.LogFormatterSciNotation())
                axis.set_minor_formatter(ticker.NullFormatter())
            ax.grid(True, which="major", linestyle="-", alpha=0.6, color="gray", zorder=1)
            ax.grid(True, which="minor", linestyle=":", alpha=0.4, zorder=1)
            gtxt = f"厚み {gv} mm" if by == "H" else f"設定温度 {int(gv)} ℃"
            if cfg["show_title"]:
                ax.set_title(f"{MATS[mat]}　せん断速度と粘度の関係（{gtxt}）\n{title_model}", fontsize=ft["title"], pad=15)
            if cfg["show_xlabel"]:
                ax.set_xlabel(r"せん断速度 $\dot{\gamma}$ (1/s)", fontsize=ft["axis"], labelpad=15)
            if cfg["show_ylabel"]:
                ax.set_ylabel("樹脂粘度 (Pa·s)", fontsize=ft["axis"], labelpad=15)
            fig.tight_layout()
            base = f"{stem}_{'by-thickness' if by == 'H' else 'by-temperature'}_{(str(gv) + 'mm') if by == 'H' else (str(int(gv)) + 'C')}"
            for fmt in cfg["formats"]:
                path = os.path.join(out_dir, f"{base}.{fmt}")
                fig.savefig(path, bbox_inches="tight", dpi=cfg["dpi"])
                saved.append(path)
            if cfg["show"]:
                plt.show()
            plt.close(fig)
    return saved

### 手順5の部品

構造とHの扱いの定義（`layout5`・`design5`・`kernel5`）、学習と予測（`fit5`・`predict5`・`rebuild5`）、足し算の成分の予測（`component_predict`）、プロファイル尤度の計算（`prof_chain`）、P3_2 の予測（`p32_model`・`predict5_p`）、並列の仕事（`job5`）。中身は `04_補助スクリプト/step5_core.py` と同じ。

In [ ]:
# セル：手順5の部品（構造・H の扱い・プロファイル尤度・P3_2 の予測・学習の仕事）。04_補助スクリプト/step5_core.py の中身。直すときは、そのファイルを直して、ノートを作り直す
import time as _time

from scipy.linalg import cho_solve

R_GAS, TREF_K = 8.314, 483.15                     # 気体定数 [J/mol/K]、基準温度 210 ℃ [K]
HV_JA = {"1": "(1) Hを残す", "2": "(2) m＋c(Fo)に分ける", "4": "(4) P3_2（Hなし）", "14": "(1+4)", "24": "(2+4)"}
STRUCT_JA5 = {"B": "B（横・縦を別々）", "A": "A（縦だけ）", "C": "C（構造なし）"}
COL_JA = {"u": "ln γ̇", "H": "ln H", "T": "Tset", "w": "ln Fo", "p": "ln P3_2"}


def shift_ln(Tk, E):
    """ln a(T; E) = (E/R)(1/T − 1/Tref)。E は kJ/mol、Tk は K"""
    return (E * 1000.0 / R_GAS) * (1.0 / np.asarray(Tk) - 1.0 / TREF_K)


def has_p(hv):
    return "4" in hv


def layout5(struct, hv):
    """入力の列の並び cols と、(2) の足し算の分け方 (m の列番号, c の列番号)（足し算でなければ None, None）。
    列の記号：u＝ln γ̇（B・A はシフト後）、H＝ln H、T＝Tset、w＝ln Fo、p＝ln P3_2（条件平均）"""
    if struct == "C":
        base = {"1": ["u", "H", "T"], "2": ["u", "T", "w"], "4": ["u", "T", "p"], "14": ["u", "H", "T", "p"], "24": ["u", "T", "p", "w"]}
    else:
        base = {"1": ["u", "w"], "2": ["u", "w"], "4": ["u", "p"], "14": ["u", "w", "p"], "24": ["u", "p", "w"]}
    cols = base[hv]
    if hv in ("2", "24"):
        return cols, [i for i, c in enumerate(cols) if c != "w"], [cols.index("w")]
    return cols, None, None


def kernel5(struct, hv):
    """(1)(4)(1+4)：ARD Matern 5/2 ＋ 線形 ＋ ノイズ（kernel_for と同じ）。(2)(2+4)：m の部分と c の部分を足し算（それぞれ Matern ＋ 線形）＋ ノイズ"""
    cols, m, c = layout5(struct, hv)
    if m is None:
        return kernel_for(len(cols), True)

    def part(d):
        return C(1.0, (1e-3, 1e3)) * Matern(length_scale=[1.0] * d, length_scale_bounds=(1e-2, 1e2), nu=2.5) + C(1.0, (1e-3, 1e3)) * DotProduct(sigma_0=1.0, sigma_0_bounds=(1e-5, 1e3))

    return OnDims(part(len(m)), m, "m") + OnDims(part(len(c)), c, "c") + WhiteKernel(noise_level=0.1, noise_level_bounds=(NOISE_LB, 1e1))


def design5(df, v):
    """データ → (標準化の定数を決めるための入力 X0、実際の入力 Xs、出力のシフト量 ysh)。v：{"struct", "hv", "Eh", "Ev", "pcol"}。
    入力の γ̇ は条件平均（ln_gd_cm）。B・A は u = ln γ̇ + ln a(Tset; E_h)、ysh = ln a(Tset; E_v)。P3_2 は pcol の列（既定 ln_P32_cm）"""
    cols = layout5(v["struct"], v["hv"])[0]
    lng, lnH, T = df["ln_gd_cm"].to_numpy(), df["ln_H"].to_numpy(), df["Tset"].to_numpy()
    raw = {"u": lng, "H": lnH, "T": T, "w": -(lng + 3.0 * lnH)}            # ln Fo = −(ln γ̇ + 3 ln H) + 定数（シフト前の γ̇ で計算）
    if has_p(v["hv"]):
        raw["p"] = df[v.get("pcol", "ln_P32_cm")].to_numpy()
    shifted = dict(raw)
    ysh = np.zeros(len(df))
    if v["struct"] != "C":
        Tk = T + 273.15
        shifted["u"] = lng + shift_ln(Tk, v["Eh"])
        ysh = shift_ln(Tk, v["Ev"])
    return np.column_stack([raw[c] for c in cols]), np.column_stack([shifted[c] for c in cols]), ysh


def fit5(v, tr, restarts, theta0=None):
    """学習。標準化・中心化は E に依存しない定数（シフト前の入力の平均・標準偏差、ln η の平均）。theta0：カーネルの初期値（前の格子点の解）"""
    X0, Xs, ysh = design5(tr, v)
    mu_x, sd_x = X0.mean(axis=0), X0.std(axis=0)
    y = tr["ln_eta"].to_numpy()
    y0 = float(y.mean())
    k = kernel5(v["struct"], v["hv"])
    if theta0 is not None:
        k = k.clone_with_theta(theta0)
    gp = GaussianProcessRegressor(kernel=k, n_restarts_optimizer=restarts, random_state=SEED).fit((Xs - mu_x) / sd_x, y - ysh - y0)
    return {"gp": gp, "mu_x": mu_x, "sd_x": sd_x, "y0": y0, "v": v}


def rebuild5(v, tr, theta):
    """学習済みのカーネルの値 theta から、学習をやり直さずに予測器を作り直す（最適化なし。並列の仕事から theta だけを受け取る）"""
    X0, Xs, ysh = design5(tr, v)
    mu_x, sd_x = X0.mean(axis=0), X0.std(axis=0)
    y = tr["ln_eta"].to_numpy()
    y0 = float(y.mean())
    gp = GaussianProcessRegressor(kernel=kernel5(v["struct"], v["hv"]).clone_with_theta(theta), optimizer=None).fit((Xs - mu_x) / sd_x, y - ysh - y0)
    return {"gp": gp, "mu_x": mu_x, "sd_x": sd_x, "y0": y0, "v": v}


def predict5(fit, df):
    """予測平均、予測標準偏差（ノイズ込み）、潜在関数の標準偏差、ノイズの標準偏差"""
    X0, Xs, ysh = design5(df, fit["v"])
    mu, sd = fit["gp"].predict((Xs - fit["mu_x"]) / fit["sd_x"], return_std=True)
    noise = fit["gp"].kernel_.k2.noise_level
    return mu + fit["y0"] + ysh, sd, np.sqrt(np.maximum(sd ** 2 - noise, 0.0)), float(np.sqrt(noise))


def component_predict(fit, df, part):
    """足し算のモデル (2)(2+4) の成分 part（"m" か "c"）の事後平均と事後標準偏差（ln η の単位。定数は m と c の間で入れ替えられる）"""
    gp = fit["gp"]
    k = gp.kernel_
    kc = k.k1.k1 if part == "m" else k.k1.k2
    assert kc.tag == part
    X0, Xs, ysh = design5(df, fit["v"])
    Xt = (Xs - fit["mu_x"]) / fit["sd_x"]
    Ks = kc(Xt, gp.X_train_)
    mean = Ks @ gp.alpha_
    var = kc.diag(Xt) - np.einsum("ij,ji->i", Ks, cho_solve((gp.L_, True), Ks.T))
    return mean, np.sqrt(np.maximum(var, 0.0))


def kernel_row5(fit, tr=None):
    """ハイパーパラメータの一覧（部品ごとの振幅・長さスケール・線形項、ノイズ、対数周辺尤度、下限・上限への張り付き）"""
    v, gp = fit["v"], fit["gp"]
    k = gp.kernel_
    cols = layout5(v["struct"], v["hv"])[0]
    if hasattr(k.k1.k1, "tag"):
        parts = [("m", k.k1.k1.kernel, k.k1.k1.dims), ("c", k.k1.k2.kernel, k.k1.k2.dims)]
    else:
        parts = [("f", k.k1, list(range(len(cols))))]
    row = {"Eh": v.get("Eh", np.nan), "Ev": v.get("Ev", np.nan), "LML": float(gp.log_marginal_likelihood_value_), "noise_var": float(k.k2.noise_level),
           "noise_sd": float(np.sqrt(k.k2.noise_level)), "noise_at_lower_bound": bool(k.k2.noise_level < NOISE_LB * 1.05)}
    at_bound = False
    for name, kk, dims in parts:
        prod, lin = kk.k1, kk.k2
        ls = np.atleast_1d(prod.k2.length_scale)
        at_bound = at_bound or bool(np.any(ls > 99) or np.any(ls < 0.0105))
        row[f"amp_{name}"] = float(np.sqrt(prod.k1.constant_value))
        row[f"lin_amp_{name}"] = float(np.sqrt(lin.k1.constant_value))
        row[f"lin_sigma0_{name}"] = float(lin.k2.sigma_0)
        for d, l in zip(dims, ls):
            row[f"ls_{name}_{cols[d]}"] = float(l)
    row["ls_at_bound"] = at_bound
    return row


def cond_table5(df):
    """条件ごとの平均（ln η、ln γ̇、ln P3_2）。尤度の比較に使う"""
    g = df.groupby("cond", sort=False)
    t = g.agg(Tset=("Tset", "first"), H=("H", "first"), v=("v", "first"), ln_eta=("ln_eta", "mean"), ln_gd=("ln_gd", "mean"), ln_P32=("ln_P32", "mean"),
              n=("ln_eta", "size"), is_sens=("is_sens", "max")).reset_index()
    t["ln_H"], t["ln_gd_cm"], t["ln_P32_cm"] = np.log(t["H"]), t["ln_gd"], t["ln_P32"]
    return t


# ---------------------------------------------------------------- プロファイル尤度
def prof_chain(cdf, base_v, points, restarts):
    """格子の点 points = [(E_h, E_v), …] を順に学習して LML を返す。前の点の解を初期値にする（局所解でがたつくのを減らす）"""
    warnings.simplefilter("ignore")
    out, theta = [], None
    for eh, ev in points:
        f = fit5({**base_v, "Eh": float(eh), "Ev": float(ev)}, cdf, restarts, theta)
        theta = f["gp"].kernel_.theta
        out.append((float(f["gp"].log_marginal_likelihood_value_), float(np.sqrt(f["gp"].kernel_.k2.noise_level))))
    return out


def chains_for(struct, grid, eh_fix=None):
    """構造ごとの格子を、順に計算する点の列（chain）に分ける。A：E_v の 1 列／B：E_h ごとに E_v の列（eh_fix が数なら、E_h をその値に固定した 1 列）"""
    if struct == "A":
        return [[(0.0, float(e)) for e in grid["A"]]]
    if eh_fix is not None:
        return [[(float(eh_fix), float(e)) for e in grid["B_ev"]]]
    return [[(float(eh), float(ev)) for ev in grid["B_ev"]] for eh in grid["B_eh"]]


def profile_best(pdf, key_cols):
    """格子の点の表 → 条件（key_cols）ごとの最良の E と、LML が最大から 1.92 以内の範囲"""
    rows = []
    for key, g in pdf.groupby(key_cols):
        r = g.loc[g["LML"].idxmax()]
        inside = g[g["LML"] >= r["LML"] - 1.92]
        z = g[(g["Eh"] == 0.0) & (g["Ev"] == 0.0)]
        row = dict(zip(key_cols, key))
        diff = inside["Ev"] - inside["Eh"]
        row.update({"Eh_best": float(r["Eh"]), "Ev_best": float(r["Ev"]), "diff_best": float(r["Ev"] - r["Eh"]), "LML_best": float(r["LML"]),
                    "LML_at_E0": float(z["LML"].iloc[0]) if len(z) else np.nan, "n_in_95": int(len(inside)),
                    "Eh_95_lo": float(inside["Eh"].min()), "Eh_95_hi": float(inside["Eh"].max()), "Ev_95_lo": float(inside["Ev"].min()), "Ev_95_hi": float(inside["Ev"].max()),
                    "diff_95_lo": float(diff.min()), "diff_95_hi": float(diff.max()), "n_points": int(len(g)),
                    "at_edge": bool(r["Ev"] in (g["Ev"].min(), g["Ev"].max()) or (g["Eh"].nunique() > 1 and r["Eh"] in (g["Eh"].min(), g["Eh"].max())))})
        row["dLML_vs_E0"] = row["LML_best"] - row["LML_at_E0"]
        rows.append(row)
    return pd.DataFrame(rows)


# ---------------------------------------------------------------- P3_2 の予測（(4) の「予測した P3_2」）
def p32_model(tr):
    """条件平均の ln P3_2 を (ln γ̇, ln H, Tset) から予測する GPR（学習の条件の点で学習）"""
    ct = cond_table5(tr)
    X = ct[["ln_gd_cm", "ln_H", "Tset"]].to_numpy()
    sc = StandardScaler().fit(X)
    y0 = float(ct["ln_P32_cm"].mean())
    gp = GaussianProcessRegressor(kernel=kernel_for(3, True), n_restarts_optimizer=N_RESTARTS, random_state=SEED).fit(sc.transform(X), ct["ln_P32_cm"].to_numpy() - y0)
    return {"gp": gp, "sc": sc, "y0": y0}


def p32_predict(pm, te):
    """テストの条件ごとに ln P3_2 を予測する。条件ごとの表（cond、実測の条件平均、予測平均、予測標準偏差〔ノイズ込み〕）と、行ごとの条件の番号を返す"""
    ct = cond_table5(te)
    mu, sd = pm["gp"].predict(pm["sc"].transform(ct[["ln_gd_cm", "ln_H", "Tset"]].to_numpy()), return_std=True)
    out = pd.DataFrame({"cond": ct["cond"], "H": ct["H"], "Tset": ct["Tset"], "v": ct["v"], "actual": ct["ln_P32_cm"], "pred": mu + pm["y0"], "pred_sd": sd})
    idx = pd.Series(np.arange(len(ct)), index=ct["cond"]).loc[te["cond"]].to_numpy()
    return out, idx


def predict5_p(fit, te, pm, n_draw, seed):
    """P3_2 を予測して使う場合：P3_2 の予測分布から n_draw 回引き、そのつど ln η を予測して、平均と分散にまとめる（全分散の法則）"""
    ptab, idx = p32_predict(pm, te)
    z = np.random.RandomState(seed).standard_normal((n_draw, len(ptab)))
    mus, vs, vfs = [], [], []
    for d in range(n_draw):
        pdraw = ptab["pred"].to_numpy() + ptab["pred_sd"].to_numpy() * z[d]
        mu, sd, sdf, _ = predict5(fit, te.assign(ln_P32_cm=pdraw[idx]))
        mus.append(mu)
        vs.append(sd ** 2)
        vfs.append(sdf ** 2)
    mus = np.array(mus)
    between = mus.var(axis=0)
    return mus.mean(axis=0), np.sqrt(np.mean(vs, axis=0) + between), np.sqrt(np.mean(vfs, axis=0) + between)


# ---------------------------------------------------------------- 学習と予測の仕事（並列）
def job5(mat, ver, sname, tr, tests, v, restarts, n_draw):
    """1 つのモデル（v）を学習し、tests の各テストを予測する。戻り値は、theta と予測（受け渡しやすいように、モデルそのものは返さない）。
    P3_2 を使うモデルは、実測の P3_2（条件平均）と、予測した P3_2 の両方で評価する"""
    warnings.simplefilter("ignore")
    t0 = _time.time()
    fit = fit5(v, tr, restarts)
    out = {"theta": fit["gp"].kernel_.theta, "kernel": kernel_row5(fit), "preds": {}}
    pm = p32_model(tr) if has_p(v["hv"]) else None
    for tn, te in tests.items():
        mu, sd, sdf, _ = predict5(fit, te)
        out["preds"][(tn, "実測" if pm is not None else "―")] = (mu, sd, sdf)
        if pm is not None:
            out["preds"][(tn, "予測")] = predict5_p(fit, te, pm, n_draw, SEED)
    out["sec"] = _time.time() - t0
    return out

## 2. データの読み込みと、部品の確認

手順3d の 2 つの版の分け方（`version_sets`）で、学習データとテストデータを作る。学習データの名前は、`S_V`・`S_T`・`CV1`〜（分け方）と、`ALL`（最終モデル用の全部）。5data 版の `ALL` は `S_V` と同じ学習データなので、計算を共有する。

**部品の確認**（判定）
- `OnDims` に全部の列を渡したカーネルが、ふつうのカーネルと、カーネル行列・勾配で一致する。
- C_1 と C_4 が、手順3d・3e の R3・W2 と同じ RMSE になる（同じ入力・同じ分け方・同じ乱数なので）。

In [ ]:
# セル2：読み込みと版の確認
D = load_all()
leg = legacy_check(D)
judge(bool(leg["同じ行"].all()) or not USE_EXCLUSIONS, "一覧の「除外」を除いたデータが、これまでの no1.5-3.0 のファイルと同じ（取り直した後は違ってよい）")
VS = {(mat, ver): version_sets(mat, ver, MAIN_DATA, TEMP_HOLDOUT) for ver in DATA_VERSIONS for mat in MATS}
TRAIN_DF, TESTS, ALIAS = {}, {}, {}
for (mat, ver), vs in VS.items():
    names = []
    for s in SPLITS[ver]:
        names += [k for k in vs["splits"] if k.startswith("CV")] if s == "CV" else ([s] if s in vs["splits"] else [])
        if s != "CV" and s not in vs["splits"]:
            print(f"（{MATS[mat]}・{ver} 版に、分け方 {s} はない。飛ばす）")
    for n in names:
        TRAIN_DF[(mat, ver, n)], TESTS[(mat, ver, n)] = vs["splits"][n]
    if ver == "5data" and "S_V" in names:
        ALIAS[(mat, ver)] = "S_V"                    # 5data 版の最終モデル（ALL）は、S_V と同じ学習データ
    else:
        TRAIN_DF[(mat, ver, "ALL")], TESTS[(mat, ver, "ALL")] = vs["pool"], {}
    pool = vs["pool"]
    print(f"{MATS[mat]}・{ver} 版：最終モデルの学習 {len(pool)} 行 {pool['cond'].nunique()} 条件。分け方 {names}"
          + "".join(f"\n    {n}：学習 {len(TRAIN_DF[(mat, ver, n)])} 行、テスト " + "、".join(f"{k} {len(t)} 行" for k, t in TESTS[(mat, ver, n)].items()) for n in names[:3]))


def tr_of(mat, ver, sname):
    """学習データ。5data 版の ALL は S_V と同じ"""
    return TRAIN_DF[(mat, ver, ALIAS[(mat, ver)] if (sname == "ALL" and (mat, ver) in ALIAS) else sname)]


def sets_of(mat, ver):
    """計算する学習データの名前（5data 版の ALL は S_V と共有するので入れない）。最終モデルは ALL（別名を含む）"""
    return [n for (m, v, n) in TRAIN_DF if m == mat and v == ver]


def mname(st, hv):
    return f"{st}_{hv}"


print("構造:", STRUCTS, "／H の扱い:", H_VARIANTS)

In [ ]:
# セル3：部品の確認（OnDims、C_1 = R3、C_4 = W2）
rng = np.random.RandomState(0)
mat0, ver0 = list(VS)[0]
tr0 = tr_of(mat0, ver0, [n for n in sets_of(mat0, ver0) if not n.startswith("CV")][0])
X0_ = StandardScaler().fit_transform(tr0[["ln_gd_cm", "ln_H", "Tset"]].to_numpy())
k_plain = kernel_for(3, True)
k_on = OnDims(C(1.0, (1e-3, 1e3)) * Matern(length_scale=[1.0] * 3, length_scale_bounds=(1e-2, 1e2), nu=2.5) + C(1.0, (1e-3, 1e3)) * DotProduct(sigma_0=1.0, sigma_0_bounds=(1e-5, 1e3)),
              [0, 1, 2], "f") + WhiteKernel(noise_level=0.1, noise_level_bounds=(NOISE_LB, 1e1))
th_ = rng.uniform(-1, 1, len(k_plain.theta))
(Ka, Ga), (Kb, Gb) = k_plain.clone_with_theta(th_)(X0_, eval_gradient=True), k_on.clone_with_theta(th_)(X0_, eval_gradient=True)
judge(max(np.abs(Ka - Kb).max(), np.abs(Ga - Gb).max()) < 1e-8, f"OnDims（全部の列）とふつうのカーネルの、カーネル行列・勾配の最大の差が 1e-8 未満（差 {max(np.abs(Ka - Kb).max(), np.abs(Ga - Gb).max()):.1e}）")
# 学習データが最小の分け方で、C_1 と R3、C_4 と W2 を比べる
mat1, ver1, n1 = min(((m, v, n) for (m, v, n) in TRAIN_DF if n != "ALL" and TESTS[(m, v, n)]), key=lambda k: len(TRAIN_DF[k]))
tr1, te1 = TRAIN_DF[(mat1, ver1, n1)], list(TESTS[(mat1, ver1, n1)].values())[0]
MODELS["R3"] = {"kind": "gpr", "cols": ["ln_gd_cm", "ln_H", "Tset"], "linear": True}
MODELS["W2"] = {"kind": "gpr", "cols": ["ln_gd_cm", "Tset", "ln_P32_cm"], "linear": True}
for hv_, key_ in [("1", "R3"), ("4", "W2")]:
    mu_a = predict5(fit5({"struct": "C", "hv": hv_, "pcol": "ln_P32_cm"}, tr1, N_RESTARTS), te1)[0]
    mu_b = predict_model(key_, fit_model(key_, tr1, N_RESTARTS), te1)[0]
    ra, rb = float(np.sqrt(np.mean((te1["ln_eta"].to_numpy() - mu_a) ** 2))), float(np.sqrt(np.mean((te1["ln_eta"].to_numpy() - mu_b) ** 2)))
    judge(abs(ra - rb) <= 0.003, f"C_{hv_} の RMSE が、{key_} と ±0.003 で一致（{MATS[mat1]}・{ver1} 版・{n1}：C_{hv_} {ra:.4f}、{key_} {rb:.4f}）")

## 3. プロファイル尤度（条件平均）

各学習データ（S_V・S_T・CV の各分割・ALL）、各構造（A・B）、各 H の扱いについて、E の格子の各点で GPR を学習し、LML を求める。**LML が大きいほど、データに合っている**。H の扱いが違うと入力の次元が違うので、LML の絶対値は扱いの間では比べられない（扱いごとに、最大との差で見る）。

- A：E_v の 1 次元の格子（E_h = 0）。
- B：(E_h, E_v) の 2 次元の格子。**斜めの帯**になる（差 E_v − E_h だけが決まる）。
- 最良の点と、LML が最大から 1.92 以内の範囲（おおよその 95% 区間）を `best_E.csv` に保存する。
- 判定：①格子の端に張り付いていないか。②E = 0（温度の効果なし）より LML が 2 以上大きいか。③学習データを変えても、A の E_v と B の差 E_v − E_h が 5 kJ/mol 以内か。④B が A より LML が 2 以上大きいか。

版ごとにセルを分けてある（長いので）。

In [ ]:
# セル4：プロファイルの計算の関数（並列）
PROF_STRUCTS = [s for s in STRUCTS if s != "C"]
PROF_LIST = []


def grid_for(sname):
    cv = sname.startswith("CV")
    return {"A": E_GRID_A, "B_eh": EH_GRID_CV if cv else EH_GRID, "B_ev": EV_GRID_CV if cv else EV_GRID}


def run_chains(tasks, stage):
    """tasks：[(キー(材料, 版, 学習データ, 構造, 扱い), 条件平均の表, 基本の指定, 格子の点の列)] → 結果の表"""
    tasks = sorted(tasks, key=lambda t: -len(t[3]))
    out = Parallel(n_jobs=N_JOBS)(delayed(prof_chain)(t[1], t[2], t[3], N_RESTARTS_PROFILE) for t in tasks)
    rows = []
    for (key, _, _, pts), res in zip(tasks, out):
        for (eh, ev), (lml, nz) in zip(pts, res):
            rows.append({"mat": key[0], "version": key[1], "set": key[2], "struct": key[3], "hv": key[4], "Eh": round(eh, 6), "Ev": round(ev, 6), "LML": lml, "noise_sd": nz, "stage": stage})
    return pd.DataFrame(rows)


KEYC = ["mat", "version", "set", "struct", "hv"]


def profile_version(ver):
    """版 ver の全材料・全学習データのプロファイルを計算する（粗い格子 → 最良の点の近くの細かい格子）。結果を PROF_LIST に足す"""
    t0 = time.time()
    CD = {(mat, ver, n): cond_table5(tr_of(mat, ver, n)) for mat in MATS for n in sets_of(mat, ver)}
    tasks = []
    for (mat, _, n), cdf in CD.items():
        for st in PROF_STRUCTS:
            for hv in H_VARIANTS:
                for pts in chains_for(st, grid_for(n), B_EH_FIX):
                    tasks.append(((mat, ver, n, st, hv), cdf, {"struct": st, "hv": hv, "pcol": "ln_P32_cm"}, pts))
    coarse = run_chains(tasks, "粗い格子")
    print(f"{ver} 版：粗い格子 {len(coarse)} 点、{(time.time() - t0) / 60:.1f} 分")
    best0 = profile_best(coarse, KEYC)
    tasks = []
    for _, r in best0.iterrows():
        key = (r["mat"], r["version"], r["set"], r["struct"], r["hv"])
        cdf = CD[(r["mat"], r["version"], r["set"])]
        bv = {"struct": r["struct"], "hv": r["hv"], "pcol": "ln_P32_cm"}
        if r["struct"] == "A":
            tasks.append((key, cdf, bv, [(0.0, float(e)) for e in r["Ev_best"] + np.arange(-FINE_A, FINE_A + 0.01, 0.5)]))
        else:
            half = FINE_B_CV if r["set"].startswith("CV") else FINE_B
            ehs = [B_EH_FIX] if B_EH_FIX is not None else list(r["Eh_best"] + np.arange(-half, half + 0.01, 1.0))
            for eh in ehs:
                tasks.append((key, cdf, bv, [(float(eh), float(e)) for e in r["Ev_best"] + np.arange(-half, half + 0.01, 1.0)]))
    fine = run_chains(tasks, "細かい格子")
    allp = pd.concat([coarse, fine], ignore_index=True).drop_duplicates(subset=KEYC + ["Eh", "Ev"], keep="first")
    PROF_LIST.append(allp)
    print(f"{ver} 版：細かい格子を足して、全部で {len(allp)} 点、{(time.time() - t0) / 60:.1f} 分")

In [ ]:
# セル5：プロファイルの計算（5data 版）
if "5data" in DATA_VERSIONS and PROF_STRUCTS:
    profile_version("5data")

In [ ]:
# セル6：プロファイルの計算（Alldata 版。CV の 10 分割を含むので、1 時間ほどかかる）
if "Alldata" in DATA_VERSIONS and PROF_STRUCTS:
    profile_version("Alldata")

In [ ]:
# セル7：最良の E と区間、判定
if PROF_STRUCTS:
    prof_df = pd.concat(PROF_LIST, ignore_index=True)
    prof_df.insert(0, "material", prof_df["mat"].map(MATS))
    prof_df.drop(columns="mat").to_csv(os.path.join(OUT_DIR, "profile_likelihood.csv"), index=False, encoding="utf-8-sig")
    best_df = profile_best(prof_df, KEYC)
    best_df.insert(0, "material", best_df["mat"].map(MATS))
    best_df.drop(columns="mat").to_csv(os.path.join(OUT_DIR, "best_E.csv"), index=False, encoding="utf-8-sig")
else:
    prof_df, best_df = pd.DataFrame(), pd.DataFrame()


def best_row(mat, ver, sname, st, hv):
    """最良の E の行（5data 版の ALL は S_V の行）"""
    sname = ALIAS[(mat, ver)] if (sname == "ALL" and (mat, ver) in ALIAS) else sname
    return best_df[(best_df["mat"] == mat) & (best_df["version"] == ver) & (best_df["set"] == sname) & (best_df["struct"] == st) & (best_df["hv"] == hv)].iloc[0]


if len(best_df):
    show =best_df[best_df["set"].isin(["S_V", "S_T", "ALL"]) & ~best_df["set"].str.startswith("CV")].copy()
    show["Eh"], show["Ev"] = show["Eh_best"], show["Ev_best"]
    print("【最良の E（kJ/mol）。diff＝E_v − E_h、diff_95＝LML が最大から 1.92 以内の範囲の差。dLML_vs_E0＝E = 0 との LML の差】")
    cols_show = ["material", "version", "set", "struct", "hv", "Eh_best", "Ev_best", "diff_best", "diff_95_lo", "diff_95_hi", "dLML_vs_E0", "at_edge"]
    print(show[cols_show].round(2).to_string(index=False))
    for ver in DATA_VERSIONS:
        for mat in MATS:
            n_ = MATS[mat]
            g = best_df[(best_df["mat"] == mat) & (best_df["version"] == ver)]
            full = g[~g["set"].str.startswith("CV")]
            judge(not g["at_edge"].any() if len(g) else True, f"{n_}・{ver} 版：最良の E が格子の端に張り付いていない（張り付いた: {int(g['at_edge'].sum())} / {len(g)} 件。CV の狭い格子で張り付いたら、EH_GRID_CV・EV_GRID_CV を広げる）")
            for st in PROF_STRUCTS:
                h1 = full[(full["struct"] == st) & (full["hv"] == H_VARIANTS[0])]
                judge(bool((h1["dLML_vs_E0"] >= 2.0).all()), f"{n_}・{ver} 版・{st}（H の扱い {H_VARIANTS[0]}）：E = 0 より LML が 2 以上大きい（差 {h1['dLML_vs_E0'].round(1).tolist()}）。温度の効果がデータで支持される")
                for hv in H_VARIANTS:
                    s = full[(full["struct"] == st) & (full["hv"] == hv)]
                    col = "Ev_best" if st == "A" else "diff_best"
                    sp_ = float(s[col].max() - s[col].min()) if len(s) > 1 else 0.0
                    if hv in ("1", "2") or hv == H_VARIANTS[0]:
                        judge(sp_ <= 5.0, f"{n_}・{ver} 版・{st}（扱い {hv}）：{'E_v' if st == 'A' else '差 E_v − E_h'} が、学習データ（{', '.join(s['set'])}）を変えても 5 kJ/mol 以内（{', '.join(f'{x:.1f}' for x in s[col])}）")
            if "A" in PROF_STRUCTS and "B" in PROF_STRUCTS:
                ms_ = "S_V" if ver == "5data" else "ALL"
                for hv in H_VARIANTS:
                    a_, b_ = full[(full["struct"] == "A") & (full["hv"] == hv) & (full["set"] == ms_)], full[(full["struct"] == "B") & (full["hv"] == hv) & (full["set"] == ms_)]
                    if len(a_) and len(b_):
                        d_ = float(b_["LML_best"].iloc[0] - a_["LML_best"].iloc[0])
                        judge(d_ >= 2.0, f"{n_}・{ver} 版・扱い {hv}：B の LML が A より 2 以上大きい（差 {d_:.1f}）なら、横のシフトも必要")

## 4. 学習と評価

3 構造 × 5 通りのモデルを、最良の E（その学習データのもの）で、ショットごとに学習し直して評価する。入力の γ̇ は条件平均。

- 評価の分け方：5data 版は **S_V**（5data → 3data。低速側・高速側）と **S_T**（190・230 ℃ → 210 ℃。5data の 210 ℃ と、3data の 210 ℃〔S_T3〕）。Alldata 版は **CV**（条件単位の 10 分割。全分割の予測をまとめて評価）と **S_T**（テストは 5data 版と同じ行）、設定すれば **S_Vall**。
- (4) を含むモデルは、P3_2 を「実測」と「予測」の 2 通りで評価する（冒頭の説明）。
- 指標：log η の RMSE、R²、95% 予測区間の被覆率、NLPD。`noise_sd`（モデルのノイズの標準偏差）と `pure_sd`（同じ条件内のショット間のばらつき＝純粋誤差）も並べる。入力が条件平均なので、ノイズは純粋誤差と同程度かそれ以上になるはず。
- 結果は `metrics.csv`、`kernels.csv`（ハイパーパラメータと、下限・上限への張り付き）。

In [ ]:
# セル8：学習の仕事の関数
RES5 = {}


def eval_version(ver):
    """版 ver の全材料・全学習データ・全モデルを並列で学習して、予測を RES5 に入れる"""
    t0 = time.time()
    jobs = []
    for mat in MATS:
        for n in sets_of(mat, ver):
            tr = TRAIN_DF[(mat, ver, n)]
            for st in STRUCTS:
                for hv in H_VARIANTS:
                    v = {"struct": st, "hv": hv, "pcol": "ln_P32_cm"}
                    if st != "C":
                        r = best_row(mat, ver, n, st, hv)
                        v["Eh"], v["Ev"] = float(r["Eh_best"]), float(r["Ev_best"])
                    jobs.append(((mat, ver, n, st, hv), tr, TESTS[(mat, ver, n)], v, N_RESTARTS_CV if n.startswith("CV") else N_RESTARTS))
    jobs.sort(key=lambda j: -len(j[1]))
    out = Parallel(n_jobs=N_JOBS)(delayed(job5)(j[0][0], j[0][1], j[0][2], j[1], j[2], j[3], j[4], P32_DRAWS) for j in jobs)
    for j, o in zip(jobs, out):
        RES5[j[0]] = o
        RES5[j[0]]["v"] = j[3]
    for mat in MATS:
        if (mat, ver) in ALIAS:
            for st in STRUCTS:
                for hv in H_VARIANTS:
                    RES5[(mat, ver, "ALL", st, hv)] = RES5[(mat, ver, ALIAS[(mat, ver)], st, hv)]
    print(f"{ver} 版：{len(jobs)} 回の学習を並列（{N_JOBS}）で計算：{(time.time() - t0) / 60:.1f} 分")

In [ ]:
# セル9：学習（5data 版）
if "5data" in DATA_VERSIONS:
    eval_version("5data")

In [ ]:
# セル10：学習（Alldata 版。CV の 10 分割を含むので、1 時間ほどかかる）
if "Alldata" in DATA_VERSIONS:
    eval_version("Alldata")

In [ ]:
# セル11：指標とハイパーパラメータの表
FIT_CACHE = {}


def get_fit(mat, ver, sname, st, hv):
    """学習済みの予測器（並列の仕事が返した theta から、最適化なしで作り直す）"""
    key = (mat, ver, sname, st, hv)
    if key not in FIT_CACHE:
        o = RES5[key]
        FIT_CACHE[key] = rebuild5(o["v"], tr_of(mat, ver, sname), o["theta"])
    return FIT_CACHE[key]


KROWS, acc, nzs = [], {}, {}
for (mat, ver, sname, st, hv), o in RES5.items():
    if sname == "ALL" and (mat, ver) in ALIAS:
        continue
    tr = TRAIN_DF[(mat, ver, sname)]
    KROWS.append({"material": MATS[mat], "version": ver, "split": sname, "struct": st, "hv": hv, "model": mname(st, hv), "n_train": len(tr), **o["kernel"],
                  "pure_sd_ln_eta": within_stats(tr)["sd_ln_eta"], "fit_sec": o["sec"]})
    for (tn, psrc), (mu, sd, sdf) in o["preds"].items():
        te = TESTS[(mat, ver, sname)][tn]
        split = "CV" if tn.startswith("CV") else tn
        acc.setdefault((mat, ver, split, st, hv, psrc), []).append(pd.DataFrame({"y": te["ln_eta"].to_numpy(), "mu": mu, "sd": sd, "sdf": sdf, "H": te["H"].to_numpy(), "Tset": te["Tset"].to_numpy(),
                                                                                  "v": te["v"].to_numpy(), "cond": te["cond"].to_numpy(), "group": te["group"].to_numpy(), "is_sens": te["is_sens"].to_numpy()}))
        nzs.setdefault((mat, ver, split, st, hv), {})[sname] = (o["kernel"]["noise_sd"], within_stats(tr)["sd_ln_eta"])
kern_df = pd.DataFrame(KROWS)
kern_df.to_csv(os.path.join(OUT_DIR, "kernels.csv"), index=False, encoding="utf-8-sig")
PRED5, mrows = {}, []
for (mat, ver, split, st, hv, psrc), parts in acc.items():
    d = pd.concat(parts, ignore_index=True)
    if split == "CV":
        d["group"] = np.where(d["v"] <= 5, "低速側", np.where(d["v"] >= 280, "高速側", "内側"))
    PRED5[(mat, ver, split, st, hv, psrc)] = d
    subs = {"全体": np.ones(len(d), bool), **{g: (d["group"] == g).to_numpy() for g in d["group"].unique() if g != "全体"}}
    if d["is_sens"].any():
        subs.update({s + SENS_SUFFIX: m & ~d["is_sens"].to_numpy() for s, m in list(subs.items())})
    nz = np.array(list(nzs[(mat, ver, split, st, hv)].values()))
    for sub, m in subs.items():
        if m.sum() < 2:
            continue
        ev = evaluate(d["y"].to_numpy()[m], d["mu"].to_numpy()[m], d["sd"].to_numpy()[m])
        mrows.append({"material": MATS[mat], "mat": mat, "version": ver, "split": split, "subset": sub, "struct": st, "hv": hv, "model": mname(st, hv), "psrc": psrc,
                      "n_test": int(m.sum()), "n_cond_test": int(d["cond"][m].nunique()), **ev, "noise_sd": float(nz[:, 0].mean()), "pure_sd": float(nz[:, 1].mean())})
met_df = pd.DataFrame(mrows)
met_df.drop(columns="mat").to_csv(os.path.join(OUT_DIR, "metrics.csv"), index=False, encoding="utf-8-sig")
PANELS = {"5data": [("S_V", "低速側", "S_V 低速側"), ("S_V", "高速側", "S_V 高速側"), ("S_T", "全体", "S_T（210 ℃）"), ("S_T3", "全体", "S_T3（3data の 210 ℃）")],
          "Alldata": [("CV", "全体", "CV 全体"), ("CV", "低速側", "CV 低速側"), ("CV", "高速側", "CV 高速側"), ("S_T", "全体", "S_T（210 ℃）"), ("S_T3", "全体", "S_T3（3data の 210 ℃）"),
                      ("S_Vall", "低速側", "S_Vall 低速側"), ("S_Vall", "高速側", "S_Vall 高速側")]}


def mv(mat, ver, split, st, hv, sub="全体", psrc=None, col="RMSE"):
    """指標。psrc を省くと、P3_2 を使わないモデルは "―"、使うモデルは "実測" の行"""
    psrc = psrc or ("実測" if has_p(hv) else "―")
    r = met_df[(met_df["mat"] == mat) & (met_df["version"] == ver) & (met_df["split"] == split) & (met_df["struct"] == st) & (met_df["hv"] == hv) & (met_df["subset"] == sub) & (met_df["psrc"] == psrc)]
    return float(r[col].iloc[0]) if len(r) else np.nan


def panels_of(ver):
    return [p for p in PANELS[ver] if ((met_df["version"] == ver) & (met_df["split"] == p[0]) & (met_df["subset"] == p[1])).any()]


for ver in DATA_VERSIONS:
    for mat in MATS:
        print(f"\n【{MATS[mat]}・{ver} 版】RMSE（ln η）／被覆率。P3_2 を使うモデルは、実測の P3_2（括弧の中が、予測した P3_2）")
        rows_ = {}
        for hv in H_VARIANTS:
            for st in STRUCTS:
                cells_ = {}
                for sp_, sub_, lab_ in panels_of(ver):
                    s = f"{mv(mat, ver, sp_, st, hv, sub_):.3f}/{mv(mat, ver, sp_, st, hv, sub_, col='coverage95'):.2f}"
                    if has_p(hv):
                        s += f"（{mv(mat, ver, sp_, st, hv, sub_, '予測'):.3f}）"
                    cells_[lab_] = s
                rows_[mname(st, hv)] = cells_
        print(pd.DataFrame(rows_).T.to_string())

## 5. 設定温度の効果

実測と、各モデルの予測を比べる（手順5a の図6と同じ見方）。
- **実測の比**：同じ厚み・同じ γ̇ で、190 ℃ の粘度 ÷ 230 ℃ の粘度（感度確認の条件は除く）。
- **モデルの比**：5data＋3data の 210 ℃ のショットについて、ほかの入力はそのまま、Tset だけを 190 ℃・230 ℃ にして予測した粘度の比。最終モデル（ALL）を使う。P3_2 を使うモデルは、P3_2 を実測の条件平均に固定する。

実測は、「厚いほど温度の効果が大きい」傾向（0.5 mm で 1.09 倍、2.0 mm で 1.2 倍ほど）。縦だけシフトする A は、厚みによらず一定の比になる。

In [ ]:
# セル12：設定温度の効果
H_LEVELS = [0.5, 1.0, 1.5, 2.0]


def observed_ratio(mat):
    d = pool_of(mat, MAIN_DATA)
    d = d[~d["is_sens"]]
    cm = d.groupby(["Tset", "H", "v"]).agg(lg=("ln_gd", "mean"), le=("ln_eta", "mean")).reset_index()
    rows = []
    for h in H_LEVELS:
        a, c = cm[(cm["H"] == h) & (cm["Tset"] == 190)], cm[(cm["H"] == h) & (cm["Tset"] == 230)].sort_values("lg")
        ins = ((a["lg"] >= c["lg"].min()) & (a["lg"] <= c["lg"].max())).to_numpy()
        r = np.exp(a["le"].to_numpy()[ins] - np.interp(a["lg"].to_numpy()[ins], c["lg"], c["le"]))
        rows.append({"material": MATS[mat], "H": h, "observed": float(np.median(r)), "n": int(ins.sum())})
    return pd.DataFrame(rows)


obs5 = pd.concat([observed_ratio(m) for m in MATS], ignore_index=True)
obs5.to_csv(os.path.join(OUT_DIR, "observed_ratio.csv"), index=False, encoding="utf-8-sig")
erows = []
for ver in DATA_VERSIONS:
    for mat in MATS:
        base = pool_of(mat, MAIN_DATA)
        base = base[(base["Tset"] == 210) & ~base["is_sens"]]
        lo, hi = base.copy(), base.copy()
        lo["Tset"], hi["Tset"] = 190.0, 230.0
        for st in STRUCTS:
            for hv in H_VARIANTS:
                fit = get_fit(mat, ver, "ALL", st, hv)
                r = np.exp(predict5(fit, lo)[0] - predict5(fit, hi)[0])
                for h, g in base.assign(r=r).groupby("H"):
                    erows.append({"material": MATS[mat], "version": ver, "model": mname(st, hv), "struct": st, "hv": hv, "H": h, "ratio_190_230": float(np.median(g["r"]))})
eff5 = pd.DataFrame(erows)
eff5.to_csv(os.path.join(OUT_DIR, "tset_effect.csv"), index=False, encoding="utf-8-sig")
print("実測の 190/230 ℃ の粘度比（同じ γ̇、厚みごと）")
print(obs5.pivot(index="material", columns="H", values="observed").round(3).to_string())
for ver in DATA_VERSIONS:
    print(f"\nモデルの比（{ver} 版の最終モデル、Tset だけを 190 → 230 ℃）")
    print(eff5[eff5["version"] == ver].pivot_table(index=["material", "model"], columns="H", values="ratio_190_230").round(3).to_string())
for ver in DATA_VERSIONS:
    for mat in MATS:
        e = eff5[(eff5["version"] == ver) & (eff5["material"] == MATS[mat])]
        for st in STRUCTS:
            r_ = e[(e["struct"] == st) & (e["hv"] == "1")]["ratio_190_230"]
            if len(r_):
                judge(bool((r_ > 1.0).all()), f"{MATS[mat]}・{ver} 版・{st}_1：Tset だけを 190 → 230 ℃ にした予測の粘度比が、全厚みで 1 を超える（範囲 {r_.min():.2f}〜{r_.max():.2f}、実測 {obs5[obs5['material'] == MATS[mat]]['observed'].min():.2f}〜{obs5[obs5['material'] == MATS[mat]]['observed'].max():.2f}）")

## 6. (2) の分解：材料の部分 m と、厚み・冷却の補正 c(Fo)

H の扱い (2) では、ln η = ln a(E_v) ＋ m ＋ c を、足し算のカーネルで表した。最終モデル（ALL）で、2 つの成分の事後平均と 95% の帯を取り出す。
- **m（材料の部分）**：せん断速度（と温度）の関数。Cross-WLF にあたる。図では、定数 y0・縦シフト ln a(E_v)・c の平均の分を足して、「粘度 η_m [Pa·s]」として、設定温度ごとに描く。
- **c（厚み・冷却の補正）**：ln Fo の関数。学習の条件での平均を 0 にそろえてある。0 に近いほど補正が小さい。点は学習の条件で、厚みごとに色分け。
- m と c は定数だけ入れ替えられるので、c の平均を 0 にそろえた（η_m ＋ c の合計は、予測と同じ）。帯は、成分の事後分散（平均をそろえる定数の不確かさは含まない）。

判定：c の大きさ（最大と最小の差）が、温度の効果（190/230 ℃ の粘度比の対数 ≈ 0.1〜0.2）より大きい＝厚み・冷却の補正は、材料の部分とは別に、無視できない。結果は `decomposition.csv`。

In [ ]:
# セル13：(2) の分解
drows, DEC = [], {}
if "2" in H_VARIANTS:
    for ver in DATA_VERSIONS:
        for mat in MATS:
            tr = tr_of(mat, ver, "ALL")
            for st in STRUCTS:
                fit = get_fit(mat, ver, "ALL", st, "2")
                c_tr, _ = component_predict(fit, tr, "c")
                c_off = float(c_tr.mean())                                           # c の平均を 0 にそろえるための定数（m に足す）
                cdf = cond_table5(tr)
                c_c, c_s = component_predict(fit, cdf, "c")
                w_c = -(cdf["ln_gd_cm"] + 3.0 * cdf["ln_H"]).to_numpy()
                for w_, h_, c_, s_ in zip(w_c, cdf["H"], c_c - c_off, c_s):
                    drows.append({"material": MATS[mat], "version": ver, "struct": st, "kind": "c_cond", "H": h_, "Tset": np.nan, "x": float(w_), "mean": float(c_), "sd": float(s_)})
                wg = np.linspace(w_c.min() - 0.2, w_c.max() + 0.2, 80)
                gdf = pd.DataFrame({"ln_gd_cm": 0.0, "ln_H": -wg / 3.0, "Tset": 210.0})
                cg, sg = component_predict(fit, gdf, "c")
                for w_, c_, s_ in zip(wg, cg - c_off, sg):
                    drows.append({"material": MATS[mat], "version": ver, "struct": st, "kind": "c_curve", "H": np.nan, "Tset": np.nan, "x": float(w_), "mean": float(c_), "sd": float(s_)})
                lg_lo, lg_hi = float(tr["ln_gd"].min()), float(tr["ln_gd"].max())
                lgg = np.linspace(lg_lo, lg_hi, 80)
                for t in [190, 210, 230]:
                    mdf = pd.DataFrame({"ln_gd_cm": lgg, "ln_H": 0.0, "Tset": float(t)})
                    mm, ms = component_predict(fit, mdf, "m")
                    ysh = design5(mdf, fit["v"])[2]
                    for x_, m_, s_ in zip(lgg, mm + fit["y0"] + ysh + c_off, ms):
                        drows.append({"material": MATS[mat], "version": ver, "struct": st, "kind": "m_curve", "H": np.nan, "Tset": float(t), "x": float(x_), "mean": float(m_), "sd": float(s_)})
                DEC[(mat, ver, st)] = c_off
dec_df = pd.DataFrame(drows)
dec_df.to_csv(os.path.join(OUT_DIR, "decomposition.csv"), index=False, encoding="utf-8-sig")
if len(dec_df):
    for ver in DATA_VERSIONS:
        for mat in MATS:
            for st in STRUCTS:
                cc = dec_df[(dec_df["material"] == MATS[mat]) & (dec_df["version"] == ver) & (dec_df["struct"] == st) & (dec_df["kind"] == "c_cond")]["mean"]
                eo = obs5[obs5["material"] == MATS[mat]]["observed"]
                judge(float(cc.max() - cc.min()) > float(np.log(eo.max())), f"{MATS[mat]}・{ver} 版・{st}_2：c(Fo) の大きさ（最大 − 最小 = {float(cc.max() - cc.min()):.2f}）が、温度の効果（ln の実測比の最大 {float(np.log(eo.max())):.2f}）より大きい＝厚み・冷却の補正が、材料の部分と分けて必要")
else:
    print("H の扱い (2) を含めていないので、分解は飛ばす")

## 7. (4) の P3_2：実測と予測

P3_2 を使うモデルの評価は、P3_2 が分かっている場合（実測）と、事前に分からず予測して使う場合（予測）で、どれだけ違うか。
- **P3_2 の予測の精度**：検証の条件の ln P3_2（条件平均）を、(ln γ̇, ln H, Tset) から予測したときの RMSE（`p32_pred.csv`）。
- **ln η の RMSE の比**：予測した P3_2 を使ったときの RMSE ÷ 実測の P3_2 を使ったときの RMSE。1 に近いほど、P3_2 が分からなくても良い。大きければ、(4) の良さは「P3_2 が分かっているおかげ」。
- **H を残したモデルとの比較**：(1+4) が (1) より良くなるか。良くならなければ、H があれば P3_2 は要らない（手順3b の R3P と同じ）。

In [ ]:
# セル14：P3_2 の予測の精度と、実測／予測の比
p32rows = []
for (mat, ver, sname), tests in TESTS.items():
    if sname == "ALL" or sname not in [n for n in sets_of(mat, ver)] or not tests:
        continue
    pm = p32_model(TRAIN_DF[(mat, ver, sname)])
    for tn, te in tests.items():
        ptab, _ = p32_predict(pm, te)
        ptab.insert(0, "split", "CV" if tn.startswith("CV") else tn)
        ptab.insert(0, "version", ver)
        ptab.insert(0, "material", MATS[mat])
        p32rows.append(ptab)
p32_df = pd.concat(p32rows, ignore_index=True) if p32rows else pd.DataFrame()
if len(p32_df):
    p32_df.to_csv(os.path.join(OUT_DIR, "p32_pred.csv"), index=False, encoding="utf-8-sig")
    s = p32_df.assign(e=p32_df["pred"] - p32_df["actual"]).groupby(["material", "version", "split"])["e"].agg(lambda x: float(np.sqrt(np.mean(x ** 2)))).rename("RMSE_ln_P32")
    print("【ln P3_2（条件平均）の予測の RMSE。ln η の RMSE と比べて見る】")
    print(s.round(3).to_string())
    P4 = [hv for hv in H_VARIANTS if has_p(hv)]
    rr = []
    for ver in DATA_VERSIONS:
        for mat in MATS:
            for sp_, sub_, lab_ in panels_of(ver):
                for st in STRUCTS:
                    for hv in P4:
                        a, b = mv(mat, ver, sp_, st, hv, sub_, "実測"), mv(mat, ver, sp_, st, hv, sub_, "予測")
                        rr.append({"material": MATS[mat], "version": ver, "panel": lab_, "model": mname(st, hv), "RMSE_actual": a, "RMSE_predicted": b, "ratio": b / a,
                                   "RMSE_with_H_(1)": mv(mat, ver, sp_, st, "1", sub_) if "1" in H_VARIANTS else np.nan})
    p4_df = pd.DataFrame(rr)
    p4_df.to_csv(os.path.join(OUT_DIR, "p32_actual_vs_predicted.csv"), index=False, encoding="utf-8-sig")
    print("\n【P3_2 を予測して使ったときの ln η の RMSE ÷ 実測の P3_2 のときの RMSE】（1 に近いほど、P3_2 が分からなくても同じ）")
    print(p4_df.pivot_table(index=["material", "version", "panel"], columns="model", values="ratio").round(2).to_string())
    for ver in DATA_VERSIONS:
        for mat in MATS:
            q = p4_df[(p4_df["version"] == ver) & (p4_df["material"] == MATS[mat]) & (p4_df["model"].str.endswith("_4"))]
            judge(bool((q["ratio"] <= 1.2).all()), f"{MATS[mat]}・{ver} 版：(4) の予測した P3_2 での RMSE が、実測の P3_2 の 1.2 倍以内（最大 {q['ratio'].max():.2f}）。超えるなら、(4) の良さは「P3_2 が分かっているおかげ」")
else:
    print("(4) を含むモデルがないか、テストがないので飛ばす")

## 8. 本命と対案の比較（ユーザー決定、2026-10-12）

本命（`MAIN_STRUCT`、既定は B）と対案（`ALT_STRUCT`、既定は A）を、基準 C と、H の扱いごとに並べる。結果は `main_vs_alt.csv`。

- **E**：最終モデル（ALL）の最良の E（E_h・E_v、差）と、LML、E = 0 との LML の差。
- **検証の指標**：RMSE（ln η）・被覆率・NLPD（版ごとの検証の分け方）。
- **設定温度の効果**：190/230 ℃ の粘度比（厚みごと）と、実測。

判定：①本命・対案の RMSE が、基準 C の 1.1 倍以内か（同等なら、構造がデータで支持された）。②(2) の RMSE が (1) の 1.1 倍以内か（分けても精度が落ちない）。③(1+4) が (1) より 10% 以上は良くならないか（H があれば P3_2 は要らない）。④被覆率が 0.90〜0.99。⑤ノイズが純粋誤差の 0.8 倍以上。

In [ ]:
# セル15：本命と対案の比較
ROLES = [("本命", MAIN_STRUCT), ("対案", ALT_STRUCT), ("基準", "C")]
cmp_rows = []
for ver in DATA_VERSIONS:
    pn = panels_of(ver)
    for mat in MATS:
        for hv in H_VARIANTS:
            for role, st in ROLES:
                if st not in STRUCTS:
                    continue
                row = {"material": MATS[mat], "version": ver, "hv": hv, "role": role, "struct": st, "model": mname(st, hv)}
                if st != "C" and len(best_df):
                    b = best_row(mat, ver, "ALL", st, hv)
                    row.update({"Eh": b["Eh_best"], "Ev": b["Ev_best"], "diff": b["diff_best"], "LML": b["LML_best"], "dLML_vs_E0": b["dLML_vs_E0"], "diff_95_lo": b["diff_95_lo"], "diff_95_hi": b["diff_95_hi"]})
                for sp_, sub_, lab_ in pn:
                    for metric in ["RMSE", "coverage95", "NLPD"]:
                        row[f"{metric}_{lab_}"] = mv(mat, ver, sp_, st, hv, sub_, col=metric)
                    if has_p(hv):
                        row[f"RMSE_{lab_}（P3_2 予測）"] = mv(mat, ver, sp_, st, hv, sub_, "予測")
                e = eff5[(eff5["version"] == ver) & (eff5["material"] == MATS[mat]) & (eff5["model"] == mname(st, hv))].sort_values("H")
                for h, r_ in zip(e["H"], e["ratio_190_230"]):
                    row[f"ratio_{h}"] = float(r_)
                cmp_rows.append(row)
cmp_df = pd.DataFrame(cmp_rows)
cmp_df.to_csv(os.path.join(OUT_DIR, "main_vs_alt.csv"), index=False, encoding="utf-8-sig")
for ver in DATA_VERSIONS:
    for mat in MATS:
        n_ = MATS[mat]
        pn = panels_of(ver)
        t = cmp_df[(cmp_df["version"] == ver) & (cmp_df["material"] == n_)]
        print(f"\n【{n_}・{ver} 版】E（ALL）と、検証の RMSE（P3_2 を使うモデルは実測の P3_2）")
        print(t[["hv", "role", "struct", "Eh", "Ev", "diff", "dLML_vs_E0"] + [f"RMSE_{lab}" for _, _, lab in pn]].round(3).to_string(index=False))
        for hv in H_VARIANTS:
            for role, st in ROLES[:2]:
                if st not in STRUCTS or "C" not in STRUCTS:
                    continue
                bad = [lab for sp_, sub_, lab in pn if mv(mat, ver, sp_, st, hv, sub_) > 1.1 * mv(mat, ver, sp_, "C", hv, sub_)]
                judge(not bad, f"{n_}・{ver} 版・扱い {hv}：{role} {st} の RMSE が、基準 C の 1.1 倍以内（超えた検証: {bad or 'なし'}）")
        for st in STRUCTS:
            if "1" in H_VARIANTS and "2" in H_VARIANTS:
                bad = [lab for sp_, sub_, lab in pn if mv(mat, ver, sp_, st, "2", sub_) > 1.1 * mv(mat, ver, sp_, st, "1", sub_)]
                judge(not bad, f"{n_}・{ver} 版・{st}：(2) の RMSE が (1) の 1.1 倍以内（超えた検証: {bad or 'なし'}）。分けても精度が落ちない")
            if "1" in H_VARIANTS and "14" in H_VARIANTS:
                good = [lab for sp_, sub_, lab in pn if mv(mat, ver, sp_, st, "14", sub_) < 0.9 * mv(mat, ver, sp_, st, "1", sub_)]
                judge(not good, f"{n_}・{ver} 版・{st}：(1+4) が (1) より 10% 以上は良くならない（良くなった検証: {good or 'なし'}）。H があれば P3_2 は要らない")
        m_ = met_df[(met_df["material"] == n_) & (met_df["version"] == ver) & (met_df["subset"] == "全体") & (met_df["psrc"] != "予測")]
        judge(bool(m_["coverage95"].between(0.90, 0.99).all()), f"{n_}・{ver} 版：被覆率が全モデルで 0.90〜0.99（外れたモデル: {sorted(set(m_[~m_['coverage95'].between(0.90, 0.99)]['model']))}）")
        judge(bool((m_["noise_sd"] >= 0.8 * m_["pure_sd"]).all()), f"{n_}・{ver} 版：ノイズの標準偏差が、純粋誤差の 0.8 倍以上（下回ったモデル: {sorted(set(m_[m_['noise_sd'] < 0.8 * m_['pure_sd']]['model']))}）")

## 9. 図

- 図1：A のプロファイル尤度（E_v と、最大との LML の差。H の扱いごとの線）
- 図2：B の 2 次元プロファイル（E_h × E_v。斜めの帯）
- 図3：マスターカーブ（換算後のデータ）
- 図4：設定温度ごとの予測曲線と予測区間（B・A・C、H の扱いごと）
- 図5：評価指標の比較
- 図6：設定温度の効果（190/230 ℃ の粘度比）
- 図7：(2) の分解（m と c）
- 図8：(4) の P3_2 の実測と予測

すべて `save_fig` で保存する（通常の PNG と、`highres/` の 600 dpi）。

In [ ]:
# セル16：図の共通の設定
ST_COL = {"B": "tab:purple", "A": "tab:blue", "C": "k"}
HV_COL = {"1": "tab:red", "2": "tab:green", "4": "tab:blue", "14": "tab:orange", "24": "tab:brown"}
T_COL = {190: "tab:blue", 210: "tab:green", 230: "tab:red"}
H_COL = {0.5: "#d62728", 1.0: "#1f77b4", 1.5: "#f0b400", 2.0: "#2ca02c"}
FIG_HV = [hv for hv in FIG_VARIANTS if hv in H_VARIANTS]


def points_pool(mat, ver):
    """曲線の図の点に使うデータ。5data 版：5data＋3data（3data は検証の条件）／Alldata 版：Alldata"""
    return pool_of(mat, MAIN_DATA) if ver == "5data" else D[(mat, "Alldata", MAIN_DATA)]


def train_conds_of(mat, ver):
    return set(tr_of(mat, ver, "ALL")["cond"])


def figname(stem, *parts):
    return "_".join([stem, *[str(p) for p in parts]])


def main_set(ver):
    return "S_V" if ver == "5data" else "ALL"

In [ ]:
# セル17：図1 A のプロファイル尤度（E_v と、最大との LML の差）
if len(prof_df) and "A" in PROF_STRUCTS:
    for ver in DATA_VERSIONS:
        sets_ = ["S_V", "S_T", "ALL"] if ver == "5data" else ["S_T", "ALL"]
        fig, axes = plt.subplots(len(MATS), len(sets_), figsize=(5.6 * len(sets_), 4.6 * len(MATS)), squeeze=False)
        for i, mat in enumerate(MATS):
            for j, sname in enumerate(sets_):
                ax = axes[i, j]
                sn = ALIAS[(mat, ver)] if (sname == "ALL" and (mat, ver) in ALIAS) else sname
                for hv in H_VARIANTS:
                    g = prof_df[(prof_df["mat"] == mat) & (prof_df["version"] == ver) & (prof_df["set"] == sn) & (prof_df["struct"] == "A") & (prof_df["hv"] == hv)].sort_values("Ev")
                    if g.empty:
                        continue
                    ax.plot(g["Ev"], g["LML"] - g["LML"].max(), "-o", ms=2.5, lw=1.2, color=HV_COL[hv], label=HV_JA[hv])
                    b = g.loc[g["LML"].idxmax()]
                    ax.plot([b["Ev"]], [0.0], "*", ms=13, color=HV_COL[hv], mec="k")
                ax.axhline(-1.92, color="gray", ls="--", lw=0.8)
                ax.axvline(0, color="gray", lw=0.8)
                ax.set_ylim(-30, 2)
                ax.set_title(f"{MATS[mat]}・{ver} 版：{sname} の学習", fontsize=10)
                ax.set_xlabel("A の E_v [kJ/mol]")
                ax.set_ylabel("対数周辺尤度 − その最大")
                ax.grid(alpha=0.3)
                if i == 0 and j == 0:
                    ax.legend(fontsize=7, loc="lower right")
        fig.suptitle(f"{ver} 版：A のプロファイル尤度。★＝最良、破線＝最大から 1.92（おおよその 95% 区間の境）、縦線＝E = 0（温度の効果なし）", fontsize=11)
        fig.tight_layout()
        save_fig(fig, OUT_DIR, figname("fig1_profile_A", ver))
        plt.show()

In [ ]:
# セル18：図2 B の 2 次元プロファイル（斜めの帯）
if len(prof_df) and "B" in PROF_STRUCTS:
    for ver in DATA_VERSIONS:
        for mat in MATS:
            sn = ALIAS[(mat, ver)] if (main_set(ver) == "ALL" and (mat, ver) in ALIAS) else main_set(ver)
            fig, axes = plt.subplots(1, len(H_VARIANTS), figsize=(4.6 * len(H_VARIANTS), 4.8), squeeze=False)
            for j, hv in enumerate(H_VARIANTS):
                ax = axes[0, j]
                g = prof_df[(prof_df["mat"] == mat) & (prof_df["version"] == ver) & (prof_df["set"] == sn) & (prof_df["struct"] == "B") & (prof_df["hv"] == hv)]
                if g.empty:
                    continue
                b = g.loc[g["LML"].idxmax()]
                w_ = 10.0                                   # 最良の点の近く（±10 kJ/mol）を拡大して見せる。全体は、粗い格子の点が、ほぼ一様に低い（LML が最大より 40 以上低い）
                gz = g[(abs(g["Eh"] - b["Eh"]) <= w_) & (abs(g["Ev"] - b["Ev"]) <= w_)]
                z = np.clip(gz["LML"].to_numpy() - b["LML"], -40, 0)
                sc_ = ax.tricontourf(gz["Eh"], gz["Ev"], z, levels=np.linspace(-40, 0, 21), cmap="viridis")
                ax.scatter(gz["Eh"], gz["Ev"], s=5, color="k", alpha=0.35)
                ins = gz["LML"] >= b["LML"] - 1.92
                ax.scatter(gz.loc[ins, "Eh"], gz.loc[ins, "Ev"], s=40, facecolors="none", edgecolors="w", linewidths=1.2)
                ax.plot([b["Eh"]], [b["Ev"]], "*", ms=14, color="r", mec="w", label=f"最良（{b['Eh']:.0f}, {b['Ev']:.0f}）")
                lim = [b["Eh"] - w_ - 2, b["Eh"] + w_ + 2]
                ax.plot(lim, lim, "k--", lw=0.8, label="E_h = E_v")
                ax.plot(lim, [x + (b["Ev"] - b["Eh"]) for x in lim], "r:", lw=0.8, label=f"E_v − E_h = {b['Ev'] - b['Eh']:.0f}")
                ax.axvline(0, color="gray", lw=0.6, ls=":")
                ax.set_xlim(b["Eh"] - w_, b["Eh"] + w_)
                ax.set_ylim(b["Ev"] - w_, b["Ev"] + w_)
                ax.set_title(HV_JA[hv], fontsize=10)
                ax.set_xlabel("横方向 E_h [kJ/mol]")
                ax.set_ylabel("縦方向 E_v [kJ/mol]")
                ax.legend(fontsize=6, loc="upper left")
                if j == len(H_VARIANTS) - 1:
                    fig.colorbar(sc_, ax=ax, label="LML − 最大")
            fig.suptitle(f"{MATS[mat]}・{ver} 版（{sn} の学習）：B の対数周辺尤度（最良の点の ±10 kJ/mol を拡大）。白丸：最大から 1.92 以内。尤度の山は E_v − E_h が一定の斜めの帯", fontsize=11)
            fig.tight_layout()
            save_fig(fig, OUT_DIR, figname("fig2_profile_B", ver, mat))
            plt.show()

In [ ]:
# セル19：図3 マスターカーブ（ALL の学習データを、最良の E で換算）
if len(best_df):
    hv_m = "1" if "1" in H_VARIANTS else H_VARIANTS[0]
    cols_m = [("N", "シフトなし（E = 0）")] + [(s, f"{s}：{STRUCT_JA5[s]}") for s in PROF_STRUCTS[::-1]]
    for ver in DATA_VERSIONS:
        for mat in MATS:
            cdf = cond_table5(tr_of(mat, ver, "ALL"))
            fig, axes = plt.subplots(4, len(cols_m), figsize=(4.8 * len(cols_m), 13), sharey="row", squeeze=False)
            for j, (name, lab) in enumerate(cols_m):
                if name == "N":
                    v = {"struct": "A", "hv": hv_m, "Eh": 0.0, "Ev": 0.0}
                else:
                    r = best_row(mat, ver, "ALL", name, hv_m)
                    v = {"struct": name, "hv": hv_m, "Eh": float(r["Eh_best"]), "Ev": float(r["Ev_best"])}
                X0_, Xs_, ysh_ = design5(cdf, v)
                for i, h in enumerate(H_LEVELS):
                    ax = axes[i, j]
                    for t, col in T_COL.items():
                        m = ((cdf["H"] == h) & (cdf["Tset"] == t)).to_numpy()
                        ax.scatter(Xs_[m, 0], cdf["ln_eta"].to_numpy()[m] - ysh_[m], s=16, color=col, alpha=0.8, label=f"{t} ℃")
                    ax.grid(alpha=0.3)
                    if j == 0:
                        ax.set_ylabel(f"厚み {h} mm\n" + r"換算後の $\ln\eta$（縦シフトを引いた値）")
                    if i == 0:
                        ax.set_title(f"{lab}\n(E_h, E_v) = ({v['Eh']:.1f}, {v['Ev']:.1f}) kJ/mol", fontsize=10)
                    if i == 3:
                        ax.set_xlabel(r"換算後の $\ln\dot{\gamma}$ ＋ 横シフト")
            axes[0, 0].legend(fontsize=8)
            fig.suptitle(f"{MATS[mat]}・{ver} 版：マスターカーブ（条件平均。H の扱い {hv_m}。3 つの温度の点が重なっていれば、その構造でシフトできている）", fontsize=11)
            fig.tight_layout(rect=(0, 0, 1, 0.98))
            save_fig(fig, OUT_DIR, figname("fig3_master", ver, mat))
            plt.show()

In [ ]:
# セル20：図4 設定温度ごとの予測曲線と予測区間（最終モデル ALL。列が構造、行が厚み）
for ver in DATA_VERSIONS:
    for mat in MATS:
        pool, tconds = points_pool(mat, ver), train_conds_of(mat, ver)
        for hv in FIG_HV:
            fig, axes = plt.subplots(4, len(STRUCTS), figsize=(4.8 * len(STRUCTS), 14), sharex="row", sharey="row", squeeze=False)
            for j, st in enumerate(STRUCTS):
                fit = get_fit(mat, ver, "ALL", st, hv)
                for i, h in enumerate(H_LEVELS):
                    ax = axes[i, j]
                    for t, col in T_COL.items():
                        sub = pool[(pool["H"] == h) & (pool["Tset"] == t)]
                        if sub.empty:
                            continue
                        cm = sub.groupby("cond").agg(lg=("ln_gd", "mean"), le=("ln_eta", "mean"), sens=("is_sens", "max")).reset_index()
                        lg = np.linspace(sub["ln_gd"].min(), sub["ln_gd"].max(), 120)
                        mu, sd, _, _ = predict5(fit, hr_inputs(sub, lg, h, t))
                        ax.plot(np.exp(lg), np.exp(mu), color=col, lw=1.4)
                        ax.fill_between(np.exp(lg), np.exp(mu - 1.96 * sd), np.exp(mu + 1.96 * sd), color=col, alpha=0.12)
                        is_tr = cm["cond"].isin(tconds).to_numpy()
                        ax.scatter(np.exp(cm["lg"][is_tr]), np.exp(cm["le"][is_tr]), s=14, color=col, alpha=0.8, zorder=3)
                        ax.scatter(np.exp(cm["lg"][~is_tr]), np.exp(cm["le"][~is_tr]), s=26, facecolors="none", edgecolors=col, marker="s", linewidths=1.2, zorder=3)
                        se = cm["sens"].to_numpy().astype(bool)
                        ax.scatter(np.exp(cm["lg"][se]), np.exp(cm["le"][se]), s=45, color="k", marker="x", zorder=4)
                    ax.set_xscale("log")
                    ax.set_yscale("log")
                    ax.xaxis.set_major_locator(LogLocator(base=10.0, subs=(1.0, 2.0, 5.0)))      # 範囲が狭い軸でも、目盛りの数字が重ならないように
                    ax.xaxis.set_major_formatter(FuncFormatter(lambda x_, _: f"{x_:g}"))
                    ax.xaxis.set_minor_formatter(NullFormatter())
                    ax.grid(alpha=0.3, which="both")
                    if j == 0:
                        ax.set_ylabel(f"厚み {h} mm\n" + r"$\eta$ [Pa·s]")
                    if i == 0:
                        v = fit["v"]
                        ax.set_title(mname(st, hv) + ("" if st == "C" else f"\n(E_h, E_v) = ({v['Eh']:.1f}, {v['Ev']:.1f})"), fontsize=10)
                    if i == 3:
                        ax.set_xlabel(r"$\dot{\gamma}$ [1/s]")
            handles = [plt.Line2D([], [], color=c, label=f"{t} ℃") for t, c in T_COL.items()]
            handles += [plt.Line2D([], [], color="gray", marker="o", ls="", label="学習の条件平均"), plt.Line2D([], [], color="gray", marker="s", mfc="none", ls="", label="学習に使わない条件"),
                        plt.Line2D([], [], color="k", marker="x", ls="", label="感度確認の条件")]
            axes[0, 0].legend(handles=handles, fontsize=7, loc="upper right")
            note = "　※ (4) を含む曲線の P3_2 は、同じ厚み・温度の条件平均を内挿した値（予測のときには分からない）" if has_p(hv) else ""
            fig.suptitle(f"{MATS[mat]}・{ver} 版・H の扱い {HV_JA[hv]}：設定温度ごとの予測曲線（線：予測、帯：95% 予測区間）{note}", fontsize=11)
            fig.tight_layout(rect=(0, 0, 1, 0.98))
            save_fig(fig, OUT_DIR, figname("fig4_curves", ver, mat, hv))
            plt.show()

In [ ]:
# セル21：図5 評価指標の比較（RMSE と NLPD）
for ver in DATA_VERSIONS:
    pn = panels_of(ver)
    labels = [(st, hv, ps) for hv in H_VARIANTS for st in STRUCTS for ps in (["実測", "予測"] if has_p(hv) else ["―"])]
    fig, axes = plt.subplots(2, len(MATS), figsize=(9.5 * len(MATS), 9.5), squeeze=False)
    colors = ["tab:blue", "tab:red", "tab:purple", "tab:brown", "tab:green", "tab:orange", "gray"]
    for j, mat in enumerate(MATS):
        for i, (metric, ylab) in enumerate([("RMSE", r"RMSE（$\ln\eta$）"), ("NLPD", "NLPD（小さいほど良い）")]):
            ax = axes[i, j]
            w = 0.8 / len(pn)
            for k, (sp_, sub_, lab_) in enumerate(pn):
                ax.bar(np.arange(len(labels)) + (k - (len(pn) - 1) / 2) * w, [mv(mat, ver, sp_, st, hv, sub_, ps if ps != "―" else None, metric) for st, hv, ps in labels], w, color=colors[k], label=lab_)
            ax.set_xticks(np.arange(len(labels)))
            ax.set_xticklabels([mname(st, hv) + {"実測": "(実)", "予測": "(予)", "―": ""}[ps] for st, hv, ps in labels], rotation=75, fontsize=7)
            ax.set_title(f"{MATS[mat]}・{ver} 版：{ylab}（入力の " + r"$\dot{\gamma}$" + " は条件平均。(実)＝実測の P3_2、(予)＝予測した P3_2）", fontsize=9)
            ax.grid(alpha=0.3, axis="y")
            if metric == "RMSE":
                ax.set_yscale("log")
            if i == 0 and j == 0:
                ax.legend(fontsize=7)
    fig.tight_layout()
    save_fig(fig, OUT_DIR, figname("fig5_metrics", ver))
    plt.show()

In [ ]:
# セル22：図6 設定温度の効果
MK = {"B": "P", "A": "v", "C": "s"}
LS = {"1": "-", "2": "--", "4": ":", "14": "-.", "24": (0, (3, 1, 1, 1))}
for ver in DATA_VERSIONS:
    fig, axes = plt.subplots(1, len(MATS), figsize=(8.5 * len(MATS), 5.4), sharey=True, squeeze=False)
    for ax, mat in zip(axes[0], MATS):
        o = obs5[obs5["material"] == MATS[mat]]
        ax.plot(o["H"], o["observed"], "k-o", lw=2.8, ms=9, label=r"実測（同じ $\dot{\gamma}$）", zorder=10)
        for st in STRUCTS:
            for hv in FIG_HV:
                e = eff5[(eff5["version"] == ver) & (eff5["material"] == MATS[mat]) & (eff5["model"] == mname(st, hv))]
                ax.plot(e["H"], e["ratio_190_230"], marker=MK[st], ls=LS[hv], alpha=0.85, color=ST_COL[st], label=mname(st, hv))
        ax.axhline(1.0, color="gray", lw=0.8)
        ax.set_xlabel("厚み [mm]")
        ax.set_title(f"{MATS[mat]}・{ver} 版：190 ℃ と 230 ℃ の粘度の比（Tset だけを変えた予測）")
        ax.grid(alpha=0.3)
    axes[0, 0].set_ylabel("粘度の比（190 ℃ ÷ 230 ℃）")
    axes[0, 0].legend(fontsize=7, ncol=2)
    fig.tight_layout()
    save_fig(fig, OUT_DIR, figname("fig6_tset_effect", ver))
    plt.show()

In [ ]:
# セル23：図7 (2) の分解（m と c）、図8 (4) の P3_2 の実測と予測
if len(dec_df):
    for ver in DATA_VERSIONS:
        for mat in MATS:
            fig, axes = plt.subplots(2, len(STRUCTS), figsize=(5.2 * len(STRUCTS), 9), squeeze=False)
            for j, st in enumerate(STRUCTS):
                d = dec_df[(dec_df["material"] == MATS[mat]) & (dec_df["version"] == ver) & (dec_df["struct"] == st)]
                ax = axes[0, j]
                for t, col in T_COL.items():
                    g = d[(d["kind"] == "m_curve") & (d["Tset"] == t)]
                    ax.plot(np.exp(g["x"]), np.exp(g["mean"]), color=col, lw=1.6, label=f"{t} ℃")
                    ax.fill_between(np.exp(g["x"]), np.exp(g["mean"] - 1.96 * g["sd"]), np.exp(g["mean"] + 1.96 * g["sd"]), color=col, alpha=0.15)
                ax.set_xscale("log")
                ax.set_yscale("log")
                ax.set_xlabel(r"$\dot{\gamma}$ [1/s]")
                ax.set_ylabel(r"材料の部分 $\eta_m$ [Pa·s]（c を平均 0 にそろえた）")
                ax.set_title(f"{mname(st, '2')}：m（材料の部分）", fontsize=10)
                ax.grid(alpha=0.3, which="both")
                if j == 0:
                    ax.legend(fontsize=8)
                ax = axes[1, j]
                g = d[d["kind"] == "c_curve"]
                ax.plot(g["x"], g["mean"], "k-", lw=1.6)
                ax.fill_between(g["x"], g["mean"] - 1.96 * g["sd"], g["mean"] + 1.96 * g["sd"], color="gray", alpha=0.25)
                for h, col in H_COL.items():
                    gg = d[(d["kind"] == "c_cond") & (d["H"] == h)]
                    ax.scatter(gg["x"], gg["mean"], s=14, color=col, label=f"{h} mm", zorder=3)
                ax.axhline(0.0, color="gray", lw=0.8)
                ax.set_xlabel(r"$\ln$ Fo $= -(\ln\dot{\gamma} + 3\ln H)$ ＋ 定数")
                ax.set_ylabel(r"厚み・冷却の補正 $c$（$\ln\eta$ の単位）")
                ax.set_title(f"{mname(st, '2')}：c(Fo)（厚み・冷却の補正）", fontsize=10)
                ax.grid(alpha=0.3)
                if j == 0:
                    ax.legend(fontsize=8, title="学習の条件（厚み）")
            fig.suptitle(f"{MATS[mat]}・{ver} 版：(2) の分解。上：材料の部分 m、下：厚み・冷却の補正 c（帯は成分の 95% の帯）", fontsize=11)
            fig.tight_layout(rect=(0, 0, 1, 0.97))
            save_fig(fig, OUT_DIR, figname("fig7_decomposition", ver, mat))
            plt.show()
if len(p32_df):
    for ver in DATA_VERSIONS:
        q = p32_df[(p32_df["version"] == ver) & (p32_df["split"].isin(["S_V", "CV"]))]
        if q.empty:
            q = p32_df[p32_df["version"] == ver]
        fig, axes = plt.subplots(1, len(MATS), figsize=(6.2 * len(MATS), 5.6), squeeze=False)
        for ax, mat in zip(axes[0], MATS):
            g = q[q["material"] == MATS[mat]]
            for h, col in H_COL.items():
                gg = g[g["H"] == h]
                ax.errorbar(gg["actual"], gg["pred"], yerr=1.96 * gg["pred_sd"], fmt="o", ms=4, color=col, alpha=0.7, elinewidth=0.6, label=f"{h} mm")
            lim = [g["actual"].min() - 0.2, g["actual"].max() + 0.2]
            ax.plot(lim, lim, "k--", lw=0.8)
            ax.set_xlabel(r"実測の $\ln P_{3\_2}$（条件平均、Pa）")
            ax.set_ylabel(r"予測した $\ln P_{3\_2}$（95% 区間つき）")
            ax.set_title(f"{MATS[mat]}・{ver} 版：検証の条件の P3_2（RMSE {float(np.sqrt(np.mean((g['pred'] - g['actual']) ** 2))):.3f}）", fontsize=10)
            ax.grid(alpha=0.3)
            ax.legend(fontsize=8)
        fig.suptitle("(4) で使う P3_2 の予測（(ln γ̇, ln H, Tset) から GPR で予測。検証の条件）。対角線に乗るほど、P3_2 を事前に予測できる", fontsize=10)
        fig.tight_layout()
        save_fig(fig, OUT_DIR, figname("fig8_p32_pred", ver))
        plt.show()

## 10. 判定のまとめ

OK は想定どおり、★ は想定と違う、または注意が要る点。基準は、判定の文の中に書いてある。上の各節の判定を、ここに集めて数える。

In [ ]:
# セル24：保存したファイルの一覧
print("保存したファイル:")
for f in sorted(os.listdir(OUT_DIR)):
    if os.path.isfile(os.path.join(OUT_DIR, f)):
        print("  ", os.path.normpath(os.path.join(OUT_DIR, f)))

## 11. 高解像度のグラフ（報告書・発表資料用）

せん断速度 ― 粘度の学習結果グラフを、GPR6月ノートのセル9-2 と同じ見せ方で、**600 dpi（15 × 10 インチ）**で 1 枚ずつ保存する。線は予測の平均、帯は 95% 区間、塗りつぶしの点は学習に使った条件、白抜きの点は学習に使わなかった条件（検証）、× は問題データ一覧の「感度確認」の条件。

- **モデル**：`HR_MODELS` に「構造_扱い」（例：B_1、A_2、C_4）を並べる。最終モデル（ALL）の曲線。(4) を含むモデルの曲線は、P3_2 を同じ厚み・温度の条件平均から内挿して描く（予測には使えない値。タイトルに書く）。
- **レイアウト**：`"thickness"`＝厚みごとに 1 枚（3 つの設定温度の曲線）、`"temperature"`＝設定温度ごとに 1 枚（4 つの厚みの曲線。6月のノートと同じ）。
- **変えられるもの**（セルの先頭の設定エリア）：描くモデル・材料・学習の種類、帯（予測区間／信頼区間）、点（条件平均／ショットごと）、軸の範囲、予測線の延長、図の大きさ・解像度・文字・マーカーの大きさ、タイトルと軸名の表示、保存形式（png／pdf／svg）。
- 全体を一度実行して、学習が終わった後なら、このセルだけを何度でも実行し直せる。

In [ ]:
# セル25：高解像度のグラフ（せん断速度 ― 粘度。最終モデルの予測。GPR6月ノートのセル9-2 にならう）
# ==========================================
# ▼▼ 設定エリア（グラフの見た目・文字サイズ。このセルの設定だけを直して、このセルだけを実行し直せる） ▼▼
# ==========================================
HR_VERSION = DATA_VERSIONS[-1]     # どの版のモデルか："5data"（5data で学習。3data の条件は白抜き）／"Alldata"（全条件で学習）
HR_MODELS = ["B_1", "A_1", "B_2", "A_2"]   # 描くモデル（「構造_扱い」。STRUCTS・H_VARIANTS に入れて学習したもの）
HR_MATERIALS = ["PP", "MIX"]       # 描く材料（PP：PP、MIX：粉砕材混合 PP）
HR_LAYOUTS = ["thickness", "temperature"]   # "thickness"：厚みごとに 1 枚（3 つの設定温度）／"temperature"：設定温度ごとに 1 枚（4 つの厚み。6月のノートと同じ）
HR_BAND = "predict"                # 帯："predict"＝95% 予測区間（ショット間のばらつき込み）／"latent"＝95% 信頼区間（平均の不確かさだけ）／None＝帯なし
HR_POINTS = "cond"                 # 点："cond"＝条件平均／"shot"＝ショットごと
HR_SHOW_TEST = True                # 学習に使わなかった条件（検証）を、白抜きで重ねるか
HR_MARK_SENS = True                # 問題データ一覧の「感度確認」の条件を、× で重ねるか
HR_X_RANGE = None                  # 予測線の範囲 [1/s]。None なら、各線のデータの範囲を HR_EXTEND 倍に延長。(30, 50000) のように指定すると、全部の線をその範囲に延長（外挿の確認用）
HR_EXTEND = 1.3
HR_XLIM = {"thickness": None, "temperature": (30, 50000)}   # 横軸（せん断速度 [1/s]）の表示範囲。(下限, 上限)＝固定（10 の累乗の目盛り。6月と同じ）／None＝その図のデータに合わせる（1・2・5 の目盛り）。
HR_YLIM = {"thickness": None, "temperature": (0.5, 5000)}   # 縦軸（粘度 [Pa·s]）の表示範囲。同じ書き方。レイアウトごとに変えられる（全部を同じ範囲にするなら、(30, 50000) のように書く）
HR_FIGSIZE = (15, 10)              # 図の大きさ（幅, 高さ）[インチ]
HR_DPI = 600                       # 解像度
HR_FONT = {"axis": 24, "legend": 24, "title": 24, "tick": 28}   # 軸名・凡例・タイトル・目盛り数字の文字サイズ
HR_MARKER = 200                    # 点の大きさ（条件平均のとき。面積）
HR_MARKER_SHOT = 70                # 点の大きさ（ショットごとのとき）
HR_LINE = 3.0                      # 予測線の太さ
HR_SHOW_TITLE = True               # タイトルを付けるか
HR_SHOW_XLABEL = False             # 軸名を付けるか（6月のノートと同じく、既定は付けない。資料に貼るとき、別に付ける場合のため）
HR_SHOW_YLABEL = False
HR_FORMATS = ["png"]               # 保存形式。"png"・"pdf"・"svg" から選ぶ（pdf・svg は、拡大しても粗くならない）
HR_SHOW = False                    # True なら、画面にも表示する（重い）
# ==========================================
HR_KW = dict(layouts=HR_LAYOUTS, band=HR_BAND, points=HR_POINTS, show_test=HR_SHOW_TEST, mark_sens=HR_MARK_SENS, x_range=HR_X_RANGE, extend=HR_EXTEND,
             xlim=HR_XLIM, ylim=HR_YLIM, figsize=HR_FIGSIZE, dpi=HR_DPI, font=HR_FONT, marker=HR_MARKER, marker_shot=HR_MARKER_SHOT, line=HR_LINE,
             show_title=HR_SHOW_TITLE, show_xlabel=HR_SHOW_XLABEL, show_ylabel=HR_SHOW_YLABEL, formats=HR_FORMATS, show=HR_SHOW)
HR_DIR = os.path.join(OUT_DIR, "highres")

hr_saved = []
for mat in HR_MATERIALS:
    pool, tconds = points_pool(mat, HR_VERSION), train_conds_of(mat, HR_VERSION)
    for name in HR_MODELS:
        st, hv = name.split("_")
        if (mat, HR_VERSION, "ALL", st, hv) not in RES5:
            print(f"{MATS[mat]}・{HR_VERSION} 版・{name} は、学習していないので描かない（設定の DATA_VERSIONS・STRUCTS・H_VARIANTS に入れて、上から実行し直す）")
            continue
        t0 = time.time()
        fit = get_fit(mat, HR_VERSION, "ALL", st, hv)
        v = fit["v"]
        label = f"{name}（{HV_JA[hv]}）" + ("" if st == "C" else f"　E_h = {v['Eh']:.1f}、E_v = {v['Ev']:.1f} kJ/mol") + f"（{HR_VERSION} 版）" + ("　※P3_2 は条件平均の内挿（予測には使えない）" if has_p(hv) else "")
        hr_saved += hr_flow(lambda Xg, f=fit: predict5(f, Xg)[:3], mat, MAIN_DATA, tconds, label, f"hr_{mat}_{HR_VERSION}_{name}", HR_DIR, pool=pool, **HR_KW)
        print(f"{MATS[mat]} {name}：保存したファイル 累計 {len(hr_saved)} 個（{time.time() - t0:.0f} 秒）")
judge(len(hr_saved) > 0, f"高解像度のグラフを {len(hr_saved)} 個保存した（{os.path.normpath(HR_DIR)}）。形式 {HR_FORMATS}、{HR_DPI} dpi")